# Assignment APIs tutorial

In this notebook we are using the python package "millionaire-client" to interact with the deployed application for the NLP assignment 2026.

Required files:
- Directory called "millionaire_client"
- 1 colab notebook

Both files must be saved in a directory in your Google Drive, for example:
```
gDrive_home/
├── Colab Notebooks/
│   └── NLP_assignment/
│       ├── PoliMillionaire.ipynb <-- Your notebook
│       └── millionaire_client/ <-- Directory provided
```

### Sign up procedure
Before showing you how the api work, you need to signup from a web browser.
- Paste this link into your browser [http://131.175.15.22:51111/](http://131.175.15.22:51111/) this is where the demo is deployed
- You will see a standard login/sign up screen, please click on sign up
- In the "email" field please enter your politecnico email, you are allwed to create only 1 account using the same email you registered to the NLP course
- Choose whever username/password you prefer (be creative ;))

### Game interaction

Once you signed up, you can start interacting already from the api.

First of all, let's connect your drive to this Colab Notebook

In [ ]:
from google.colab import drive
import os
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


Then we need to add our python package "millionaire_client" to the system path, so python can see it.

In [ ]:
import sys
import os

# Define the path to the directory containing your package
package_parent_dir = '/content/gdrive/MyDrive/Polimi/NLP_Project'

# Append to sys.path if it is not already present
if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

# Verify the path was added
print(sys.path)

['/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython', '/content/gdrive/MyDrive/Polimi/NLP_Project']


In [ ]:
import os, sys

package_parent_dir = "/content/gdrive/MyDrive/Polimi/NLP_Project"
package_dir = package_parent_dir + "/millionaire_client"

print("Parent exists:", os.path.exists(package_parent_dir))
print("Package exists:", os.path.exists(package_dir))
print("__init__ exists:", os.path.exists(package_dir + "/__init__.py"))
print("Files:", os.listdir(package_dir) if os.path.exists(package_dir) else "missing")

if package_parent_dir not in sys.path:
    sys.path.append(package_parent_dir)

print("Path added:", package_parent_dir in sys.path)

Parent exists: True
Package exists: True
__init__ exists: True
Files: ['__init__.py', 'base.py', 'models.py', 'game.py', 'client.py', 'exceptions.py', 'competitions.py', 'leaderboard.py', 'auth.py', '__pycache__']
Path added: True


In [ ]:
import importlib.util
import sys
import os

package_parent_dir = "/content/gdrive/MyDrive/Polimi/NLP_Project"
package_dir = os.path.join(package_parent_dir, "millionaire_client")

# Make sure parent is visible
if package_parent_dir not in sys.path:
    sys.path.insert(0, package_parent_dir)

# Load package manually
init_file = os.path.join(package_dir, "__init__.py")

spec = importlib.util.spec_from_file_location(
    "millionaire_client",
    init_file,
    submodule_search_locations=[package_dir]
)

millionaire_client = importlib.util.module_from_spec(spec)
sys.modules["millionaire_client"] = millionaire_client
spec.loader.exec_module(millionaire_client)

from millionaire_client import MillionaireClient, AuthenticationError

print("Import successful")

Import successful


Let's import the client classes

In [ ]:
from millionaire_client import MillionaireClient, AuthenticationError

You can save your password in a Colab secret (the "key" icon on the tab on the left) and import it into your notebook.

In [ ]:
from google.colab import userdata
pwd = userdata.get('poli-millionaire')

Now keep the API_URL as stated, but please change the username and password to be the ones you used during sign up session.

In [ ]:
API_URL = "http://131.175.15.22:51111/"
username = "youssef189B"
password = pwd

Now we can instantiate a MillionaireClient object and call the login method, which takes as parameters username and password.

In [ ]:
client = MillionaireClient(API_URL)
try:
    user = client.login(username, password)
    print(f"\nWelcome, {user.username}! (Role: {user.role})")
except AuthenticationError as e:
    print(f"Login failed: {e}")


Welcome, youssef189B! (Role: student)


After login, the web page is showing you different types of competitions, for each of them you can choose to play a game or to see the leaderboard. For now let's list all of the.

In [ ]:
# List available competitions
print("\n=== Available Competitions ===")
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"  {comp.id}: {comp.name} ({comp.max_levels} questions)")


=== Available Competitions ===
  0: Entertainment (15 questions)
  1: Ancient History and Politics (15 questions)
  2: Science and Nature (15 questions)
  3: Maths (15 questions)


In [ ]:
# Choose a competition ID
comp_id = 3

After choosing a competition, we can start a game! We can choose to start a game by calling `game = client.game.start(competition_id=comp_id)`. The object game is the one that is handling the game itself, we can call:
- game.current_question.text : to know the current question in text format
- game.current_level: to check the current level of difficulty of the question
- game.current_question.options: to check the possible choices we have to answer the question
- game.answer: to send to the server the answer we choose (the integer corresponding to our choice) and get the response (either correct or incorrect)

WATCH OUT! Each question has a timer, you have maximum 30 seconds to answer the question. As of now, if you exceed the maximum allowed time, there is not a "push notification". You still have to submit your answer anyway and, even though the answer was correct, you will get a TimedOut response!

In [ ]:
def play_game(game):
  # Play the game
  while game.in_progress:
      question = game.current_question
      if not question:
          print("No question available. Game may have ended.")
          break

      print(f"\n--- Level {game.current_level} ---")
      print(f"Q: {question.text}")
      print()

      for opt in question.options:
          print(f"  [{opt.id}] {opt.text}")

      # Get time remaining
      time_left = game.time_remaining
      if time_left:
          print(f"\nTime remaining: {time_left:.1f}s")

      # Get answer
      try:
          answer_input = input("\nYour answer (option ID): ").strip()
          answer_id = int(answer_input)
      except ValueError:
          print("Invalid input. Please enter a number.")
          continue

      # Submit answer
      result = game.answer(answer_id)

      if result.correct:
          print(" CORRECT!")
          if result.game_over:
              print(f"\n CONGRATULATIONS! You completed the game!")
              print(f" Final earnings: ${result.earned_amount:,.2f}")
          else:
              print(f" Earned so far: ${result.earned_amount:,.2f}")
      elif result.timed_out:
        print("TIMED OUT!")
        print(f"\n Game Over!")
        print(f" Final earnings: ${result.earned_amount:,.2f}")
      elif not result.correct:
          print(" WRONG ANSWER!")
          print(f"\n Game Over!")
          print(f" Final earnings: ${result.earned_amount:,.2f}")

  print("\n=== Game Summary ===")
  print(f"Reached Level: {game.current_level}")
  print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [ ]:
# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
print(f"Session ID: {game.session_id}")
print(f"Total number of questions: {game.state.competition.max_levels}")
print()
play_game(game)


=== Starting Game ===
Session ID: 67640
Total number of questions: 15


--- Level 1 ---
Q: If U and V are 3-dimensional subspaces of R^5, what are the possible dimensions of U ∩ V?

  [0] 1
  [1] 0 or 1
  [2] 0
  [3] 1, 2, or 3

Time remaining: 29.9s

Your answer (option ID): 3
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
Q: If A is a subset of the real line R and A contains each rational number, which of the following must be true?

  [0] If A is open, then A = R.
  [1] If A is uncountable, then A is open.
  [2] If A is closed, then A = R.
  [3] If A is uncountable, then A = R.

Time remaining: 29.9s

Your answer (option ID): 2
 CORRECT!
 Earned so far: $200.00

--- Level 3 ---
Q: We are interested in the proportion p of people who are unemployed in a large city. Eight percent of a simple random sample of 500 people are unemployed. What is the midpoint for a 95% confidence interval estimate of p?

  [0] 0.475
  [1] 0.025
  [2] 0.012
  [3] None of the above.

Time remaining: 29.

In [ ]:
# Show leaderboard position
lb = client.leaderboard.get(competition_id=comp_id, limit=10)
print(f"\n=== Leaderboard for {lb.competition.name} ===")
for i, entry in enumerate(lb.entries[:5], 1):
    marker = " <-- YOU" if entry.username == username else ""
    print(f"  {i}. {entry.username}: ${entry.score:,.2f} (Level {entry.reached_level}){marker}")


=== Leaderboard for Ancient History and Politics ===
  1. AleAssini: $1,024,000.00 (Level 15)
  2. supreme_leader: $1,024,000.00 (Level 15)
  3. luca_bordin: $1,024,000.00 (Level 15)
  4. Anonymous: $1,024,000.00 (Level 15)
  5. Jasmin: $1,024,000.00 (Level 15)


 # Phase 2 — Automatic Question Solver

In this phase, we will replace manual answer input with an automatic solver.

The goal is to:
1. Format each game question into a clean prompt.
2. Build a simple baseline solver.
3. Connect the solver to the game loop.
4. Later replace the baseline with a local open-weight LLM.

## 2.1 Question Formatter

This function converts the current game question into a clean text format that can be passed later to an LLM or any solver.

In [ ]:
def format_question(question):
    """
    Strong prompt forcing structured output.
    """

    options_text = "\n".join(
        [f"{opt.id}. {opt.text}" for opt in question.options]
    )

    return f"""
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
{question.text}

Options:
{options_text}

Answer:
""".strip()

In [ ]:
# Start the game
print("\n=== Starting Game ===")
game = client.game.start(competition_id=comp_id)
question = game.current_question

print(format_question(question))


=== Starting Game ===
Question:
What was the status of Greek in the Eastern Roman Empire during the late antiquity period?

Options:
[0] It was banned and replaced by Punic.
[1] It was used primarily for religious and literary purposes.
[2] It was only used by the military.
[3] It was completely replaced by Latin.

Answer with only the option ID.


## 2.2 Random Baseline Solver

Before using any AI model, we first create a simple random solver.

This verifies that:
- the automatic answering pipeline works
- the game loop integrates correctly
- timing works before adding heavy models

This is our sanity-check baseline.

In [ ]:
import random

def random_solver(question):
    """
    Select a random option from available choices.
    Used as baseline sanity check.
    """

    option_ids = [opt.id for opt in question.options]

    chosen_answer = random.choice(option_ids)

    return chosen_answer

## 2.3 Add Run Logging

Before adding the local LLM, we will log each question, selected answer, result, level, and time taken.

This helps us compare solvers later.

In [ ]:
import time

run_logs = []

def log_question_result(
    solver_name,
    level,
    question,
    selected_answer,
    result,
    time_taken
):
    run_logs.append({
        "solver": solver_name,
        "level": level,
        "question": question.text,
        "options": {opt.id: opt.text for opt in question.options},
        "selected_answer": selected_answer,
        "correct": result.correct,
        "timed_out": result.timed_out,
        "earned_amount": result.earned_amount,
        "time_taken_sec": round(time_taken, 2)
    })

In [ ]:
def play_game_auto(game, solver_function, solver_name="solver"):
    """
    Automatic version with logging.
    """

    while game.in_progress:

        question = game.current_question

        if not question:
            print("No question available.")
            break

        current_level = game.current_level
        print(f"\n--- Level {current_level} ---")

        formatted_q = format_question(question)
        print(formatted_q)

        # Time remaining
        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        # Start timing
        start_time = time.time()

        # Get answer
        answer_id = solver_function(question)

        # End timing
        end_time = time.time()
        time_taken = end_time - start_time

        print(f"\nSelected answer: {answer_id}")
        print(f"Decision time: {time_taken:.2f}s")

        # Submit answer
        result = game.answer(answer_id)

        # Log result
        log_question_result(
            solver_name=solver_name,
            level=current_level,
            question=question,
            selected_answer=answer_id,
            result=result,
            time_taken=time_taken
        )

        if result.correct:
            print(" CORRECT!")

            if result.game_over:
                print("\n CONGRATULATIONS!")
                print(f" Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f" Earned so far: ${result.earned_amount:,.2f}")

        elif result.timed_out:
            print(" TIMED OUT!")
            print("\n Game Over!")
            print(f" Final earnings: ${result.earned_amount:,.2f}")

        else:
            print(" WRONG ANSWER!")
            print("\n Game Over!")
            print(f" Final earnings: ${result.earned_amount:,.2f}")

    print("\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")

In [ ]:
run_logs = []  # reset logs

game = client.game.start(competition_id=comp_id)

play_game_auto(game, random_solver, solver_name="random")


--- Level 1 ---
Question:
What was the primary economic activity that contributed to the prosperity of ancient Greek city-states?

Options:
[0] Agriculture
[1] Manufacturing
[2] Slavery
[3] Trade

Answer with only the option ID.

Time remaining: 29.9s

Selected answer: 2
Decision time: 0.00s
 WRONG ANSWER!

 Game Over!
 Final earnings: $0.00

=== Game Summary ===
Reached Level: 1
Total Earnings: $0.00


In [ ]:
import pandas as pd

df_logs = pd.DataFrame(run_logs)
df_logs

,solver,level,question,options,selected_answer,correct,timed_out,earned_amount,time_taken_sec
0,random,1,What was the primary economic activity that co...,"{0: 'Agriculture', 1: 'Manufacturing', 2: 'Sla...",2,False,False,0,0.0


## 2.4 Local LLM Solver

Now we will load an open-weight model locally in Colab and use it to choose the answer.

The solver will:
1. Receive the question and options.
2. Ask the local model to return only the option ID.
3. Parse the model output into an integer.
4. Fall back to random choice if parsing fails.

In [ ]:
!pip install -q transformers accelerate sentencepiece

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

print("Model loaded:", model_name)

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model loaded: google/flan-t5-base


## 2.4.1 Improve Output Parsing

The model sometimes returns the answer text instead of the option ID.

We will improve the parser so it can handle:
- "2"
- "[2]"
- "Option 2"
- answer text like "Civic technology"

In [ ]:
import re

def parse_model_answer(response, question):
    """
    Parse model output into a valid option ID.
    Handles numeric answers and option-text answers.
    """

    response_clean = response.strip().lower()

    valid_option_ids = [opt.id for opt in question.options]

    # Case 1: direct number inside response, e.g. "2", "[2]", "Option 2"
    numbers = re.findall(r"\d+", response_clean)

    for num in numbers:
        num = int(num)
        if num in valid_option_ids:
            return num

    # Case 2: model returns option text, e.g. "Civic technology"
    for opt in question.options:
        option_text = opt.text.strip().lower()

        if option_text in response_clean or response_clean in option_text:
            return opt.id

    # Case 3: failed parsing
    return None

## 2.4.2 Local LLM Solver (FLAN-T5)

We will prompt the model to return ONLY the option ID (0, 1, 2, or 3).

If parsing fails, we fall back to a random answer.

In [ ]:
def local_llm_solver(question):
    """
    Use FLAN-T5 to choose the correct option ID.
    """

    prompt = format_question(question)

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=10
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    print(f"Model raw output: {response}")

    answer_id = parse_model_answer(response, question)

    if answer_id is not None:
        return answer_id

    print("Fallback to random")
    return random_solver(question)

In [ ]:
game = client.game.start(competition_id=comp_id)

question = game.current_question

local_llm_solver(question)

Model raw output: [2]


2

In [ ]:
run_logs = []  # reset logs

game = client.game.start(competition_id=comp_id)

play_game_auto(game, local_llm_solver, solver_name="flan_t5")


--- Level 1 ---
Question:
What term describes the region centered around the city of Babylon in central-southern Mesopotamia?

Options:
[0] Elam
[1] Assyria
[2] Babylonia
[3] Sumeria

Answer with only the option ID.

Time remaining: 29.9s
Model raw output: Babylonia

Selected answer: 2
Decision time: 1.13s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
Question:
What is the primary material used to cast Roman dodecahedra?

Options:
[0] Gold
[1] Copper alloy
[2] Iron
[3] Bronze

Answer with only the option ID.

Time remaining: 29.9s
Model raw output: Iron is the primary material used to cast Roman do

Selected answer: 2
Decision time: 1.48s
 WRONG ANSWER!

 Game Over!
 Final earnings: $100.00

=== Game Summary ===
Reached Level: 2
Total Earnings: $100.00


# Phase 3 — Wikipedia RAG Solver

This phase improves the solver by:
- retrieving raw text from Wikipedia using the question
- filtering relevant pages
- passing grounded context to the LLM
- forcing the model to answer based ONLY on retrieved data

In [ ]:
!pip install -q wikipedia

import wikipedia
import random
import torch

  Preparing metadata (setup.py) ... done


## 3.1 Search Wikipedia


In [ ]:
def search_wikipedia_pages(query, max_results=5):
    try:
        return wikipedia.search(query, results=max_results)
    except Exception as e:
        print("Wikipedia search error:", e)
        return []

## 3.2 Retrieve & Filter Context


In [ ]:
def get_filtered_wikipedia_context(question, max_pages=3, chars_per_page=1000, debug=False):

    query = question.text
    raw_titles = search_wikipedia_pages(query, max_results=5)

    # simple relevance filtering using word overlap
    q_words = set(question.text.lower().split())

    filtered_titles = []
    for t in raw_titles:
        t_words = set(t.lower().split())
        if len(q_words.intersection(t_words)) > 0:
            filtered_titles.append(t)

    # fallback if nothing matched
    if not filtered_titles:
        filtered_titles = raw_titles[:max_pages]
    else:
        filtered_titles = filtered_titles[:max_pages]

    contexts = []

    for title in filtered_titles:
        try:
            page = wikipedia.page(title, auto_suggest=False)
            text = page.content[:chars_per_page]
            contexts.append(f"Title: {title}\n{text}")
        except Exception as e:
            if debug:
                print(f"Skipped page: {title} | {e}")

    context = "\n\n".join(contexts)

    if debug:
        print("=== Search Query ===")
        print(query)

        print("\n=== Selected Titles ===")
        print(filtered_titles)

        print("\n=== Context Preview ===")
        print(context[:2000])

    return context

## 3.3 Prompt (STRICT grounding)


In [ ]:
def format_rag_prompt(question, context):

    options_text = "\n".join(
        [f"{opt.id}. {opt.text}" for opt in question.options]
    )

    return f"""
You MUST answer the question using ONLY the Wikipedia context.

Instructions:
- Find evidence in the context that supports one of the options
- Choose the option BEST supported by the context
- If none are perfect, choose the closest match
- Do NOT rely on outside knowledge

Rules:
- Answer using ONLY the option number (0, 1, 2, or 3)
- Do NOT explain
- Output only one number

Wikipedia context:
{context}

Question:
{question.text}

Options:
{options_text}

Answer:
""".strip()



## 3.4 RAG Solver


In [ ]:

def wikipedia_rag_solver(question, debug=False):

    context = get_filtered_wikipedia_context(
        question,
        max_pages=3,
        chars_per_page=1000,
        debug=debug
    )

    if not context.strip():
        print("No context found. Falling back to local LLM.")
        return local_llm_solver(question)

    prompt = format_rag_prompt(question, context)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if debug:
        print("\n=== LLM Output ===")
        print(response)

    answer_id = parse_model_answer(response, question)

    if answer_id is not None:
        return answer_id

    print("Parsing failed. Falling back to random.")
    return random.choice([opt.id for opt in question.options])

## 3.5 Test (single question)


In [ ]:

game = client.game.start(competition_id=comp_id)
question = game.current_question

print(format_question(question))

answer = wikipedia_rag_solver(question, debug=True)

print("\nSelected answer:", answer)

You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What term describes the period of ancient Egyptian history between the 16th century BC and the 11th century BC, marked by the Eighteenth, Nineteenth, and Twentieth dynasties?

Options:
0. Late Kingdom
1. New Kingdom
2. Old Kingdom
3. Middle Kingdom

Answer:
=== Search Query ===
What term describes the period of ancient Egyptian history between the 16th century BC and the 11th century BC, marked by the Eighteenth, Nineteenth, and Twentieth dynasties?

=== Selected Titles ===
['Ancient Egyptian literature', 'History of cartography', 'Art of ancient Egypt']

=== Context Preview ===
Title: Ancient Egyptian literature
Ancient Egyptian literature was written with the Egyptian language from ancient Egypt's pharaonic period until the end of Roman domination. It represents the oldest corpus 

## 3.6 Full Game Run


In [ ]:

run_logs = []

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    lambda q: wikipedia_rag_solver(q, debug=False),
    solver_name="rag_wikipedia"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What is the primary cause of the decline of the Twentieth Dynasty of Egypt?

Options:
0. Invasion by Sea Peoples
1. Loss of trade routes
2. Drought and famine
3. Civil war and succession issues

Answer:

Time remaining: 29.9s

Selected answer: 3
Decision time: 8.16s
 WRONG ANSWER!

 Game Over!
 Final earnings: $0.00

=== Game Summary ===
Reached Level: 1
Total Earnings: $0.00


## 3.3 Sentence-Level Context Filtering

We extract only relevant sentences from Wikipedia pages instead of passing full paragraphs.

This reduces noise and improves answer accuracy.

In [ ]:
import re

def extract_relevant_sentences(context, question, max_sentences=10):
    sentences = re.split(r'(?<=[.!?]) +', context)

    q_words = set(question.text.lower().split())

    option_words = set()
    for opt in question.options:
        option_words.update(opt.text.lower().split())

    keywords = q_words.union(option_words)

    scored = []

    for s in sentences:
        s_clean = s.lower().strip()

        if len(s_clean) < 20:
            continue

        words = set(s_clean.split())
        score = len(words.intersection(keywords))

        if score > 0:
            scored.append((score, s))

    scored.sort(reverse=True)

    selected = [s for _, s in scored[:max_sentences]]

    return "\n".join(selected)

In [ ]:
def wikipedia_rag_filtered_solver(question, debug=False):

    # Step 1: get full context (existing logic)
    context = get_filtered_wikipedia_context(
        question,
        max_pages=3,
        chars_per_page=1000,
        debug=debug
    )

    if not context.strip():
        return local_llm_solver(question)

    # Step 2: filter sentences (NEW)
    filtered_context = extract_relevant_sentences(context, question)

    if debug:
        print("\n=== FILTERED CONTEXT ===")
        print(filtered_context)

    # Step 3: build prompt (same function)
    prompt = format_rag_prompt(question, filtered_context)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=5,
        do_sample=False
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if debug:
        print("\n=== LLM OUTPUT ===")
        print(response)

    answer_id = parse_model_answer(response, question)

    if answer_id is not None:
        return answer_id

    return random.choice([opt.id for opt in question.options])

In [ ]:
game = client.game.start(competition_id=comp_id)
question = game.current_question

print(format_question(question))

answer = wikipedia_rag_filtered_solver(question, debug=True)

print("\nSelected answer:", answer)

You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
Which ancient scholar first published an edition of the Homeric poems in 1488?

Options:
0. Pisistratus
1. Euripides
2. Xenophanes of Colophon
3. Demetrios Chalkokondyles

Answer:
=== Search Query ===
Which ancient scholar first published an edition of the Homeric poems in 1488?

=== Selected Titles ===
['Homeric Hymns', 'Translations of the Odyssey']

=== Context Preview ===
Title: Homeric Hymns
The Homeric Hymns (Ancient Greek: Ὁμηρικοὶ ὕμνοι, romanised: Homērikoì húmnoi) are a collection of thirty-three ancient Greek hymns and one epigram. The hymns praise deities of the Greek pantheon and retell mythological stories, often involving a deity's birth, their acceptance among the gods on Mount Olympus, or the establishment of their cult. In antiquity, the hymns were generally, thoug

In [ ]:
run_logs = []

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    lambda q: wikipedia_rag_filtered_solver(q, debug=False),
    solver_name="rag_wikipedia_filtered"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What is the fundamental principle of Caesarism, as exemplified by Julius Caesar?

Options:
0. Rule by a democratic parliament
1. Rule by a charismatic strongman with a cult of personality
2. Rule by a council of elders
3. Rule by a popularly elected president

Answer:

Time remaining: 29.9s

Selected answer: 1
Decision time: 2.72s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
How does the Doric order differ from the Ionic order in terms of column design?

Options:
0. The Doric order has a taller column shaft.
1. The Doric order uses volutes in the capital.
2

## 3.4 Hybrid Wikipedia Search

We combine:
- full question search
- entity-based search

to improve retrieval quality.

In [ ]:
def generate_search_query(question):
    """
    Use LLM to generate a concise Wikipedia search query.
    """

    prompt = f"""
Convert the question into a short, focused Wikipedia search query.

Rules:
- Keep only key entities and concepts
- Focus on deterministic names
- Remove filler words
- Do NOT explain

Question:
{question.text}

Search Query:
""".strip()

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    outputs = llm_model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False
    )

    query = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    return query

In [ ]:
#- Output a short phrase (max 8 words)


In [ ]:
def format_rag_prompt(question, context):

    options_text = "\n".join(
        [f"{opt.id}. {opt.text}" for opt in question.options]
    )

    return f"""
You MUST choose the correct option based ONLY on the Wikipedia context.

Instructions:
- Compare each option with the context
- Find which option is directly supported by the context
- Select the best matching option
- Do NOT rely on outside knowledge

Rules:
- Answer using ONLY the option number (0, 1, 2, or 3)
- Do NOT explain
- Output only one number

Wikipedia context:
{context}

Question:
{question.text}

Options:
{options_text}

Answer:
""".strip()

In [ ]:
def get_filtered_wikipedia_context(question, max_pages=3, chars_per_page=1000, debug=False):

    # 🔹 ACTIVE: use full question for search (stable baseline)
    question_query = question.text

    # 🔹 FUTURE: LLM-generated query (currently disabled)
    # llm_query = generate_search_query(question)
    # titles_llm = search_wikipedia_pages(llm_query, max_results=5)

    # 🔹 CURRENT search
    titles_q = search_wikipedia_pages(question_query, max_results=7)

    # 🔹 FUTURE: hybrid merge (LLM + question)
    # raw_titles = list(dict.fromkeys(titles_llm + titles_q))

    # 🔹 CURRENT: only use question search
    raw_titles = titles_q

    # 🔹 Relevance filtering
    q_words = set(question.text.lower().split())

    filtered_titles = []
    for t in raw_titles:
        t_words = set(t.lower().split())
        if len(q_words.intersection(t_words)) > 0:
            filtered_titles.append(t)

    # 🔹 Fallback
    if not filtered_titles:
        filtered_titles = raw_titles[:max_pages]
    else:
        filtered_titles = filtered_titles[:max_pages]

    # 🔹 Retrieve context
    contexts = []

    for title in filtered_titles:
        try:
            page = wikipedia.page(title, auto_suggest=False)
            text = page.content[:chars_per_page]
            contexts.append(f"Title: {title}\n{text}")
        except Exception as e:
            if debug:
                print(f"Skipped page: {title} | {e}")

    context = "\n\n".join(contexts)

    # 🔹 Debug prints
    if debug:
        print("=== Question Query ===")
        print(question_query)

        # 🔹 FUTURE debug
        # print("\n=== LLM Search Query ===")
        # print(llm_query)

        print("\n=== Retrieved Titles ===")
        print(raw_titles)

        print("\n=== Selected Titles ===")
        print(filtered_titles)

        print("\n=== Context Preview ===")
        print(context[:2000])

    return context

In [ ]:
game = client.game.start(competition_id=comp_id)
question = game.current_question

print(format_question(question))

answer = wikipedia_rag_filtered_solver(question, debug=True)

print("\nSelected answer:", answer)

You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What is the main conclusion of mainstream Western scholars regarding the racial identity of ancient Egyptians?

Options:
0. Ancient Egyptians' racial identity is uncertain and unverifiable.
1. Ancient Egyptians were a homogeneous 'black' civilization.
2. Ancient Egyptians had a mixed population with varying skin colors.
3. Ancient Egyptians were a homogeneous 'white' civilization.

Answer:
=== Question Query ===
What is the main conclusion of mainstream Western scholars regarding the racial identity of ancient Egyptians?

=== Retrieved Titles ===
['Ancient Egyptian race controversy', 'Egyptians', 'Ancient Egypt', 'Ancient Macedonians', 'Prehistoric Egypt']

=== Selected Titles ===
['Ancient Egyptian race controversy', 'Ancient Egypt', 'Ancient Macedonians']

=== Context Preview ===


In [ ]:
run_logs = []

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    lambda q: wikipedia_rag_filtered_solver(q, debug=False),
    solver_name="rag_wikipedia_filtered"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What is Mycenae, mentioned in the article, known as in the context of Mycenaean Greek?

Options:
0. A method for deciphering Linear B
1. The name of a scholar who studied Mycenaean Greek
2. The primary location where Mycenaean Greek was spoken
3. A type of script used to write Mycenaean Greek

Answer:

Time remaining: 29.9s

Selected answer: 1
Decision time: 3.75s
 WRONG ANSWER!

 Game Over!
 Final earnings: $0.00

=== Game Summary ===
Reached Level: 1
Total Earnings: $0.00


# Phase 4 — Mistrale  + Tavily



In [ ]:
!pip install -q langchain langchain-community langchain-core tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.4/542.4 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
hugging-face-token =  userdata.get('hugging-face-token')


In [ ]:
!pip install -q transformers accelerate bitsandbytes huggingface_hub

from huggingface_hub import login

# Paste your Hugging Face token when prompted
login()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.8 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
GPU: Tesla T4


In [ ]:
# Load Mistral model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Format messages (Mistral uses chat template too)
messages = [
    {"role": "user", "content": "Who are you?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=False
)

# Decode only generated part
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


I am a large language model trained by Mistral AI. I am designed to generate human-like text based on the input I receive. I don't have the ability to have feelings, emotions


In [ ]:
def ask_mistral_mcq(prompt, max_new_tokens=20):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    return response

### Tavily Search

In [ ]:
# !pip install langchain_community

In [ ]:
import os
from langchain_community.tools.tavily_search import TavilySearchResults

# Add your Tavily API key here
os.environ["TAVILY_API_KEY"] = userdata.get('Tavili-API-Key')

search_tool = TavilySearchResults(k=3)

print("Tavily search tool initialized")

Tavily search tool initialized


/tmp/ipykernel_2567/2481028867.py:7: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(k=3)


In [ ]:
results = search_tool.invoke({
    "query": "Roman dodecahedra purpose"
})

results

[{'title': 'Roman dodecahedron - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/Roman_dodecahedron',
  'content': 'Scholars discussing the purpose of the objects almost always note the philosophical significance of the dodecahedron in classical thought, which is often believed to have a ritual purpose. It has been suggested that they might have been fortune-telling devices. This is based on the fact that most of the examples have been found at Gallo-Roman sites. Several dodecahedra were found in coin hoards, suggesting either that their owners considered them valuable objects, or that their use was connected with coins—as, for example, for easily checking that coins fit a certain diameter and were not clipped. It has also been suggested that they might have been an object to test the skill of a metalsmith, perhaps as part of a portfolio to demonstrate their capabilities to customers or as a way to [...] ## Purpose\n\nThe purpose of Roman dodecahedra has been much debated: more tha

In [ ]:
def get_tavily_context(question, max_results=3, debug=False):
    """
    Search Tavily using the question text and return raw retrieved content.
    """

    query = question.text + " correct answer explanation"

    results = search_tool.invoke({
        "query": query
    })

    # Keep top results only
    results = results[:max_results]

    contexts = []

    for i, r in enumerate(results, start=1):
        title = r.get("title", "")
        url = r.get("url", "")
        content = r.get("content", "")

        contexts.append(
            f"Source {i}\nTitle: {title}\nURL: {url}\nContent:\n{content}"
        )

    context = "\n\n".join(contexts)

    if debug:
        print("=== Tavily Query ===")
        print(query)

        print("\n=== Tavily Results ===")
        for r in results:
            print("-", r.get("title"))
            print(" ", r.get("url"))

        print("\n=== Context Preview ===")
        print(context[:2500])

    return context

In [ ]:
game = client.game.start(competition_id=comp_id)
question = game.current_question
context = get_tavily_context(question, debug=True)

=== Tavily Query ===
Which term refers to the younger and passive partner in a male homosexual relationship in ancient Greece?

=== Tavily Results ===
- Pederasty - Wikipedia
  https://en.wikipedia.org/wiki/Pederasty
- What Was Pederasty In Ancient Greece? | HistoryExtra
  https://www.historyextra.com/period/ancient-greece/pederasty-homosexuality-ancient-greece-boys-sparta-girls-plato-sappho-consent/
- ἀρσενοκοῖται–Paul’s meaning is crystal clear. — BEACON
  https://www.beaconssaministry.org/musings-of-a-queer-saint/arsenokoitai-pauls-meaning-is-clear

=== Context Preview ===
Source 1
Title: Pederasty - Wikipedia
URL: https://en.wikipedia.org/wiki/Pederasty
Content:
Pederasty in ancient Greece was a socially acknowledged romantic relationship between an adult male (the erastes) and a younger male (the eromenos), usually in

Source 2
Title: What Was Pederasty In Ancient Greece? | HistoryExtra
URL: https://www.historyextra.com/period/ancient-greece/pederasty-homosexuality-ancient-greece-

In [ ]:
# =========================================================
# Tavily RAG Solver using Mistral
# =========================================================

import random

def tavily_mistral_rag_solver(question, debug=False):
    """
    Tavily-based RAG solver using Mistral:
    - search Tavily
    - build context
    - ask Mistral
    - parse option ID
    """

    context = get_tavily_context(
        question,
        max_results=3,
        debug=debug
    )

    if not context.strip():
        print("No Tavily context found. Falling back to random.")
        return random.choice([opt.id for opt in question.options])

    options_text = "\n".join(
        [f"{opt.id}. {opt.text}" for opt in question.options]
    )

    prompt = f"""
You are answering a multiple choice question.

Return ONLY the number of the correct answer.

Valid outputs:
0
1
2
3


Do NOT output anything else.
Do NOT include text.
Do NOT include punctuation.

Rules:
- Use ONLY the provided context
- Do NOT explain
- Do NOT write words
- Think step by step silently, then output only the final number.
- Output a single number only

Context:
{context}

Question:
{question.text}

Options:
{options_text}

Answer:
""".strip()

    response = ask_mistral_mcq(prompt, max_new_tokens=10)

    if debug:
        print("\n=== Mistral Output ===")
        print(response)

    answer_id = parse_model_answer(response, question)

    if answer_id is not None:
        return answer_id

    print("Parsing failed. Falling back to random.")
    return random.choice([opt.id for opt in question.options])

### Debug

In [ ]:
game = client.game.start(competition_id=comp_id)
question = game.current_question

print(format_question(question))

answer = tavily_mistral_rag_solver(question, debug=True)

print("\nSelected answer:", answer)

You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What was the primary reason the ancient Egyptians developed Egyptian blue?

Options:
0. To produce a pigment for medical purposes
1. To decorate the exterior of their pyramids
2. To imitate the precious stones turquoise and lapis lazuli
3. To create a new form of glass

Answer:
=== Tavily Query ===
What was the primary reason the ancient Egyptians developed Egyptian blue?

=== Tavily Results ===
- Egyptian blue - Wikipedia
  https://en.wikipedia.org/wiki/Egyptian_blue
- Archaeologists Are Recreating the Long-Lost Recipe for Egyptian Blue, the World's Oldest Known Synthetic Pigment
  https://www.smithsonianmag.com/smart-news/researchers-are-recreating-the-long-lost-recipe-for-egyptian-blue-the-worlds-oldest-known-synthetic-pigment-180986778/
- Egyptian blue: the colour of technology 

### The game

In [ ]:
import time
import random

In [ ]:

import time
import random
import torch

def play_game_auto(game, solver_function, solver_name="solver"):

    while game.in_progress:

        question = game.current_question

        if not question:
            print("No question available.")
            break

        current_level = game.current_level

        print(f"\n--- Level {current_level} ---")
        print(format_question(question))

        time_left = game.time_remaining
        if time_left:
            print(f"\nTime remaining: {time_left:.1f}s")

        start_time = time.time()

        try:
            answer_id = solver_function(question)
        except Exception as e:
            print("Solver error:", e)
            answer_id = random.choice([opt.id for opt in question.options])
            print("Fallback random answer:", answer_id)

        time_taken = time.time() - start_time

        print(f"\nSelected answer: {answer_id}")
        print(f"Decision time: {time_taken:.2f}s")

        result = game.answer(answer_id)
        torch.cuda.empty_cache()
        if result.correct:
            print(" CORRECT!")

            if result.game_over:
                print("\n CONGRATULATIONS!")
                print(f" Final earnings: ${result.earned_amount:,.2f}")
            else:
                print(f" Earned so far: ${result.earned_amount:,.2f}")

        elif result.timed_out:
            print(" TIMED OUT!")
            print("\n Game Over!")
            print(f" Final earnings: ${result.earned_amount:,.2f}")

        else:
            print(" WRONG ANSWER!")
            print("\n Game Over!")
            print(f" Final earnings: ${result.earned_amount:,.2f}")

    print("\n=== Game Summary ===")
    print(f"Reached Level: {game.current_level}")
    print(f"Total Earnings: ${game.earned_amount:,.2f}")




### Experiment 1

In [ ]:
game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=tavily_mistral_rag_solver,
    solver_name="tavily_mistral_rag"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
Which term best describes the geographical regions and countries that were culturally and historically directly influenced by the ancient Greeks and Romans?

Options:
0. The Middle East
1. The Silk Road
2. The Caribbean
3. The Greco-Roman world

Answer:

Time remaining: 29.9s

Selected answer: 3
Decision time: 9.08s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
Which historical figure is known for distinguishing between diabetes mellitus and diabetes insipidus and is associated with Cappadocia?

Options:
0. Basil the Great
1. Alexander the Great
2. Aretaeus 

### Experiment 2


In [ ]:
game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=tavily_mistral_rag_solver,
    solver_name="tavily_mistral_rag"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
Which term describes the ancient Macedonian people's name origin?

Options:
0. It comes from the Dorian tribe that migrated to the region.
1. The name is of Illyrian origin.
2. The name is derived from the Greek adjective meaning 'tall' or 'slim'.
3. They were named after their king, Makedon.

Answer:

Time remaining: 29.9s

Selected answer: 2
Decision time: 9.52s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What term describes the region centered around the city of Babylon in central-southern Mesopotamia?

Options:
0. Sumeria
1. Elam
2. Assyria
3. Babyloni

# Web Search and Retrieval Benchmark

In this section, we investigate multiple retrieval approaches independently from the LLM and the game API.

The objective is to compare search/retrieval tools using fixed quiz questions collected from previous game runs.

At this stage, we do not generate answers and we do not submit anything to the competition API.

We only evaluate the quality of the retrieved information.

Retrievers to compare:
1. Wikipedia
2. DuckDuckGo Search
3. Tavily
4. SerpAPI (Google Search)

Evaluation dimensions:
- Whether the retrieved context contains the expected answer
- Whether the top result is relevant
- Number of useful documents returned
- Retrieval latency
- Amount of noise or irrelevant content

## Samples Questions

In [ ]:
retrieval_test_questions = [

    # =========================================================
    # Ancient History and Politics
    # =========================================================

    {
        "id": "AHP_Q1",
        "competition": "Ancient History and Politics",
        "question_type": "direct_entity",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Easy factual",
        "question": "What term describes the region centered around the city of Babylon in central-southern Mesopotamia?",
        "options": ["Elam", "Assyria", "Babylonia", "Sumeria"],
        "expected_answer": "Babylonia",
        "expected_option": 2,
        "keywords": ["Babylon", "central-southern Mesopotamia", "Babylonia"]
    },

    {
        "id": "AHP_Q2",
        "competition": "Ancient History and Politics",
        "question_type": "artifact_material",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Specific artifact knowledge",
        "question": "What is the primary material used to cast Roman dodecahedra?",
        "options": ["Gold", "Copper alloy", "Iron", "Bronze"],
        "expected_answer": "Copper alloy",
        "expected_option": 1,
        "keywords": ["Roman dodecahedra", "cast", "copper alloy", "bronze"]
    },

    {
        "id": "AHP_Q3",
        "competition": "Ancient History and Politics",
        "question_type": "named_historical_figure",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Named historical figure",
        "question": "Which ancient scholar first published an edition of the Homeric poems in 1488?",
        "options": [
            "Pisistratus",
            "Euripides",
            "Xenophanes of Colophon",
            "Demetrios Chalkokondyles"
        ],
        "expected_answer": "Demetrios Chalkokondyles",
        "expected_option": 3,
        "keywords": ["Homeric poems", "1488", "Demetrios Chalkokondyles"]
    },

    {
        "id": "AHP_Q4",
        "competition": "Ancient History and Politics",
        "question_type": "conceptual_definition",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Conceptual historical definition",
        "question": "What is the fundamental principle of Caesarism, as exemplified by Julius Caesar?",
        "options": [
            "Rule by a democratic parliament",
            "Rule by a charismatic strongman with a cult of personality",
            "Rule by a council of elders",
            "Rule by a popularly elected president"
        ],
        "expected_answer": "Rule by a charismatic strongman with a cult of personality",
        "expected_option": 1,
        "keywords": ["Caesarism", "charismatic strongman", "cult of personality"]
    },

    {
        "id": "AHP_Q5",
        "competition": "Ancient History and Politics",
        "question_type": "comparison_reasoning",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Architecture comparison",
        "question": "How does the Doric order differ from the Ionic order in terms of column design?",
        "options": [
            "The Doric order has a taller column shaft.",
            "The Doric order uses volutes in the capital.",
            "The Doric order has more flutes on the column shaft.",
            "The Doric order has a simpler, more plain capital."
        ],
        "expected_answer": "The Doric order has a simpler, more plain capital.",
        "expected_option": 3,
        "keywords": ["Doric order", "Ionic order", "column", "capital", "volutes"]
    },

    {
        "id": "AHP_Q6",
        "competition": "Ancient History and Politics",
        "question_type": "relationship_reasoning",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Hard relationship question",
        "question": "What is the connection between the Antikythera mechanism and the city of Rhodes?",
        "options": [
            "Rhodes was mentioned on the inscriptions of the mechanism",
            "Rhodes is where the mechanism was first discovered",
            "Rhodes was a center of astronomy and mechanical engineering",
            "Rhodes was the location of the shipwreck"
        ],
        "expected_answer": "Rhodes was a center of astronomy and mechanical engineering",
        "expected_option": 2,
        "keywords": ["Antikythera mechanism", "Rhodes", "astronomy", "mechanical engineering"]
    },

    # =========================================================
    # Entertainment
    # =========================================================

    {
        "id": "ENT_Q1",
        "competition": "Entertainment",
        "question_type": "music_terminology",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Jazz terminology",
        "question": "What style of vocal improvisation was Ella Fitzgerald famous for?",
        "options": ["Falsetto", "Yodeling", "Scat", "Beatboxing"],
        "expected_answer": "Scat",
        "expected_option": 2,
        "keywords": ["Ella Fitzgerald", "scat singing", "jazz vocal improvisation"]
    },

    {
        "id": "ENT_Q2",
        "competition": "Entertainment",
        "question_type": "music_voice_classification",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Opera terminology",
        "question": "What vocal range is typically between soprano and contralto?",
        "options": ["Tenor", "Mezzo-soprano", "Baritone", "Bass"],
        "expected_answer": "Mezzo-soprano",
        "expected_option": 1,
        "keywords": ["mezzo-soprano", "vocal range", "opera"]
    },

    {
        "id": "ENT_Q3",
        "competition": "Entertainment",
        "question_type": "celebrity_history",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Celebrity death cause",
        "question": "What was the cause of Marilyn Monroe's death?",
        "options": [
            "She accidentally overdosed on barbiturates.",
            "She died in a car accident.",
            "She suffered a heart attack.",
            "She drowned while swimming."
        ],
        "expected_answer": "She accidentally overdosed on barbiturates.",
        "expected_option": 0,
        "keywords": ["Marilyn Monroe", "death", "barbiturates"]
    },

    {
        "id": "ENT_Q4",
        "competition": "Entertainment",
        "question_type": "media_history",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Music industry impact",
        "question": "How did MTV influence the music industry in the 1980s?",
        "options": [
            "It made radio obsolete.",
            "They popularized the concept of music videos as a promotional tool.",
            "It ended the popularity of live concerts.",
            "It replaced physical album sales completely."
        ],
        "expected_answer": "They popularized the concept of music videos as a promotional tool.",
        "expected_option": 1,
        "keywords": ["MTV", "music videos", "1980s music industry"]
    },

    {
        "id": "ENT_Q5",
        "competition": "Entertainment",
        "question_type": "historical_motivation",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Celebrity historical decision",
        "question": "Why did Elvis Presley join the Army?",
        "options": [
            "To improve his public image",
            "To avoid taxes",
            "To dodge the draft during the Korean War era",
            "To become a military musician"
        ],
        "expected_answer": "To dodge the draft during the Korean War era",
        "expected_option": 2,
        "keywords": ["Elvis Presley", "Army", "draft"]
    },

    # =========================================================
    # Science and Nature
    # =========================================================

    {
        "id": "SCI_Q1",
        "competition": "Science and Nature",
        "question_type": "basic_atomic_structure",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Basic chemistry",
        "question": "Which particles determine the atomic number of an atom?",
        "options": ["electrons", "neutrons", "protons", "ions"],
        "expected_answer": "protons",
        "expected_option": 2,
        "keywords": ["atomic number", "protons", "atom"]
    },

    {
        "id": "SCI_Q2",
        "competition": "Science and Nature",
        "question_type": "instrument_definition",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Astronomy basics",
        "question": "What is the primary purpose of a telescope in astronomy?",
        "options": [
            "To capture images of the sun's surface",
            "To magnify and observe distant objects",
            "To detect radio waves from space",
            "To measure the distance between stars"
        ],
        "expected_answer": "To magnify and observe distant objects",
        "expected_option": 1,
        "keywords": ["telescope", "astronomy", "distant objects"]
    },

    {
        "id": "SCI_Q3",
        "competition": "Science and Nature",
        "question_type": "plant_biology",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Basic biology",
        "question": "Which part of a dandelion plant takes in minerals and water?",
        "options": ["flower", "root", "stem", "leaf"],
        "expected_answer": "root",
        "expected_option": 1,
        "keywords": ["dandelion", "root", "minerals", "water"]
    },

    {
        "id": "SCI_Q4",
        "competition": "Science and Nature",
        "question_type": "extinction_reasoning",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Environmental reasoning",
        "question": "The ichthyornis was a type of bird that lived over 65 million years ago. Fossils have been found in Kansas, many miles from the present-day ocean. Which cause of extinction is most likely to be found in the fossil record along with ichthyornis fossils?",
        "options": [
            "diseases causing illness",
            "changes in the weather",
            "predator animals",
            "changes in the environment"
        ],
        "expected_answer": "changes in the environment",
        "expected_option": 3,
        "keywords": ["ichthyornis", "fossils", "environmental changes"]
    },

    {
        "id": "SCI_Q5",
        "competition": "Science and Nature",
        "question_type": "pollination",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Plant-bee interaction",
        "question": "Many flowers attract bees. Bees consume nectar from the flower for energy. How does this interaction benefit the plant?",
        "options": [
            "by increasing photosynthesis",
            "by limiting growth",
            "by reducing competition",
            "by pollinating it"
        ],
        "expected_answer": "by pollinating it",
        "expected_option": 3,
        "keywords": ["bees", "flowers", "pollination"]
    },

    {
        "id": "SCI_Q6",
        "competition": "Science and Nature",
        "question_type": "photosynthesis",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Cell function",
        "question": "What is the main function of photosynthetic cells within a plant?",
        "options": [
            "to convert energy from sunlight into food energy",
            "to allow the passage of carbon dioxide into the plant",
            "to change oxygen into carbon dioxide",
            "to break down sugar into usable chemicals"
        ],
        "expected_answer": "to convert energy from sunlight into food energy",
        "expected_option": 0,
        "keywords": ["photosynthetic cells", "sunlight", "food energy"]
    },

    {
        "id": "SCI_Q7",
        "competition": "Science and Nature",
        "question_type": "chemistry_formula",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Chemical formula",
        "question": "What is the general formula for alums, such as potassium alum and sodium alum?",
        "options": [
            "KCr(SO4)2·12H2O",
            "KAl(SO4)2·12H2O",
            "XAl(SO4)2·12H2O",
            "Al2(SO4)3·nH2O"
        ],
        "expected_answer": "XAl(SO4)2·12H2O",
        "expected_option": 2,
        "keywords": ["alum", "general formula", "XAl(SO4)2·12H2O"]
    },

    # =========================================================
    # Maths
    # =========================================================

    {
        "id": "MATH_Q1",
        "competition": "Maths",
        "question_type": "linear_algebra",
        "retrieval_needed": False,
        "tool_needed": "symbolic_reasoning",
        "difficulty_note": "Subspace intersection reasoning",
        "question": "If U and V are 3-dimensional subspaces of R^5, what are the possible dimensions of U ∩ V?",
        "options": ["1", "0 or 1", "0", "1, 2, or 3"],
        "expected_answer": "1, 2, or 3",
        "expected_option": 3,
        "keywords": ["subspaces", "dimension", "intersection"]
    },

    {
        "id": "MATH_Q2",
        "competition": "Maths",
        "question_type": "real_analysis",
        "retrieval_needed": False,
        "tool_needed": "logical_reasoning",
        "difficulty_note": "Set theory / topology",
        "question": "If A is a subset of the real line R and A contains each rational number, which of the following must be true?",
        "options": [
            "If A is open, then A = R.",
            "If A is uncountable, then A is open.",
            "If A is closed, then A = R.",
            "If A is uncountable, then A = R."
        ],
        "expected_answer": "If A is closed, then A = R.",
        "expected_option": 2,
        "keywords": ["real line", "rational numbers", "closed set"]
    },

    {
        "id": "MATH_Q3",
        "competition": "Maths",
        "question_type": "statistics",
        "retrieval_needed": False,
        "tool_needed": "calculator",
        "difficulty_note": "Confidence interval reasoning",
        "question": "We are interested in the proportion p of people who are unemployed in a large city. Eight percent of a simple random sample of 500 people are unemployed. What is the midpoint for a 95% confidence interval estimate of p?",
        "options": ["0.475", "0.025", "0.012", "None of the above."],
        "expected_answer": "None of the above.",
        "expected_option": 3,
        "keywords": ["confidence interval", "sample proportion", "statistics"]
    },

    {
        "id": "MATH_Q4",
        "competition": "Maths",
        "question_type": "hypothesis_testing",
        "retrieval_needed": True,
        "tool_needed": None,
        "difficulty_note": "Statistical concepts",
        "question": "Which of the following is a true statement about hypothesis testing?",
        "options": [
            "The power of a test concerns its ability to detect an alternative hypothesis.",
            "If a hypothesis test is conducted at the 1% level, there is a 1% chance of rejecting the null hypothesis.",
            "Whether to use a one- or a two-sided test is typically decided after the data are gathered.",
            "If there is sufficient evidence to reject a null hypothesis at the 10% level, then there is sufficient evidence to reject it at the 5% level."
        ],
        "expected_answer": "The power of a test concerns its ability to detect an alternative hypothesis.",
        "expected_option": 0,
        "keywords": ["hypothesis testing", "power of a test", "statistics"]
    },

    {
        "id": "MATH_Q5",
        "competition": "Maths",
        "question_type": "geometry_constraint",
        "retrieval_needed": False,
        "tool_needed": "calculator",
        "difficulty_note": "Triangle inequality",
        "question": "The longest side of a triangle is 10. Which of the following could NOT be the lengths of the other two sides?",
        "options": ["3, 9", "5, 5", "4, 7", "9, 8"],
        "expected_answer": "5, 5",
        "expected_option": 1,
        "keywords": ["triangle inequality", "triangle sides"]
    },

    {
        "id": "MATH_Q6",
        "competition": "Maths",
        "question_type": "sampling_distribution",
        "retrieval_needed": False,
        "tool_needed": "statistical_reasoning",
        "difficulty_note": "Central limit reasoning",
        "question": "The total cholesterol level in a large population of people is strongly skewed right with a mean of 210 mg/dL and a standard deviation of 15 mg/dL. If random samples of size 16 are repeatedly drawn from this population, which of the following appropriately describes the sampling distribution of these sample means?",
        "options": [
            "The shape is approximately normal with a mean of 210 and a standard deviation of 3.75.",
            "The shape is unknown with a mean of 210 and a standard deviation of 15.",
            "The shape is somewhat skewed right with a mean of 210 and a standard deviation of 3.75.",
            "The shape is approximately normal with a mean of 210 and a standard deviation of 15."
        ],
        "expected_answer": "The shape is somewhat skewed right with a mean of 210 and a standard deviation of 3.75.",
        "expected_option": 2,
        "keywords": ["sampling distribution", "mean", "standard deviation"]
    },

    {
        "id": "MATH_Q7",
        "competition": "Maths",
        "question_type": "basic_arithmetic",
        "retrieval_needed": False,
        "tool_needed": "calculator",
        "difficulty_note": "Simple arithmetic",
        "question": "Nine bags of bird feed are in the storage room. Seventeen more bags will be delivered on Monday. Twenty-two bags will be delivered on Tuesday. Three bags will be delivered on Wednesday. Eleven bags will be delivered on Thursday. Lastly, eighteen bags will be delivered on Friday. By the end of the week, how many bags of bird feed will there be in total?",
        "options": ["9", "60", "25", "80"],
        "expected_answer": "80",
        "expected_option": 3,
        "keywords": ["addition", "arithmetic", "bird feed"]
    }

]

In [ ]:
import pandas as pd

questions_df = pd.DataFrame([
    {
        "id": q["id"],
        "competition": q["competition"],
        "difficulty_note": q["difficulty_note"],
        "question": q["question"],
        "expected_answer": q["expected_answer"]
    }
    for q in retrieval_test_questions
])

questions_df

,id,competition,difficulty_note,question,expected_answer
0,AHP_Q1,Ancient History and Politics,Easy factual,What term describes the region centered around...,Babylonia
1,AHP_Q2,Ancient History and Politics,Specific artifact knowledge,What is the primary material used to cast Roma...,Copper alloy
2,AHP_Q3,Ancient History and Politics,Named historical figure,Which ancient scholar first published an editi...,Demetrios Chalkokondyles
3,AHP_Q4,Ancient History and Politics,Conceptual historical definition,What is the fundamental principle of Caesarism...,Rule by a charismatic strongman with a cult of...
4,AHP_Q5,Ancient History and Politics,Architecture comparison,How does the Doric order differ from the Ionic...,"The Doric order has a simpler, more plain capi..."
5,AHP_Q6,Ancient History and Politics,Hard relationship question,What is the connection between the Antikythera...,Rhodes was a center of astronomy and mechanica...
6,ENT_Q1,Entertainment,Jazz terminology,What style of vocal improvisation was Ella Fit...,Scat
7,ENT_Q2,Entertainment,Opera terminology,What vocal range is typically between soprano ...,Mezzo-soprano
8,ENT_Q3,Entertainment,Celebrity death cause,What was the cause of Marilyn Monroe's death?,She accidentally overdosed on barbiturates.
9,ENT_Q4,Entertainment,Music industry impact,How did MTV influence the music industry in th...,They popularized the concept of music videos a...


In [ ]:
from dataclasses import dataclass
from typing import Optional, List
import time

@dataclass
class RetrievalResult:
    retriever: str
    question_id: str
    query: str
    title: str
    url: Optional[str]
    content: str
    rank: int
    latency_seconds: float

In [ ]:
def evaluate_retrieval_result(result, expected_answer, keywords):
    """
    Simple automatic evaluation for retrieval quality.
    This does not judge final answer correctness.
    It only checks whether retrieved text contains useful evidence.
    """
    text = f"{result.title} {result.content}".lower()

    expected_found = expected_answer.lower() in text

    keyword_hits = sum(
        1 for keyword in keywords
        if keyword.lower() in text
    )

    return {
        "expected_answer_found": expected_found,
        "keyword_hits": keyword_hits,
        "keyword_coverage": keyword_hits / len(keywords) if keywords else 0
    }

## Wikipedia

In [ ]:
!pip install wikipedia -q

  Preparing metadata (setup.py) ... done


### Safe Wikipedia retriever

In [ ]:
import wikipedia
import time
import random
import pandas as pd


def safe_call(func, max_retries=2, base_delay=1.0, jitter=0.3):
    """
    Runs unstable web calls safely using retries and short delays.
    """
    last_error = None

    for attempt in range(max_retries):
        try:
            return func()
        except Exception as e:
            last_error = e
            sleep_time = base_delay * (2 ** attempt) + random.uniform(0, jitter)
            time.sleep(sleep_time)

    raise last_error


def shorten_query(question_item, max_length=280):
    """
    Wikipedia search has a query length limit.
    If the question is too long, we use keywords instead.
    """
    question = question_item["question"]

    if len(question) <= max_length:
        return question

    return " ".join(question_item["keywords"])


def retrieve_from_wikipedia_safe(question_item, max_results=3, max_chars_per_page=1500):
    """
    Safe Wikipedia baseline retriever.

    Strategy:
    - Uses the full question when short enough.
    - Falls back to keywords when the question is too long.
    - Adds retry logic and delays to reduce request failures.
    """
    question_id = question_item["id"]
    query = shorten_query(question_item)

    start_time = time.time()

    try:
        page_titles = safe_call(
            lambda: wikipedia.search(query, results=max_results),
            max_retries=2,
            base_delay=1.0
        )
    except Exception as e:
        print(f"Wikipedia search failed for {question_id}: {e}")
        return []

    search_latency = time.time() - start_time

    results = []

    time.sleep(0.5)

    for rank, title in enumerate(page_titles, start=1):

        try:
            page = safe_call(
                lambda: wikipedia.page(title, auto_suggest=False),
                max_retries=2,
                base_delay=1.0
            )

            content = page.content[:max_chars_per_page]

            results.append(
                RetrievalResult(
                    retriever="Wikipedia Safe",
                    question_id=question_id,
                    query=query,
                    title=title,
                    url=page.url,
                    content=content,
                    rank=rank,
                    latency_seconds=search_latency
                )
            )

            time.sleep(0.5)

        except Exception as e:
            print(f"Could not read page '{title}' for {question_id}: {e}")

    return results

### Run Wikipedia safe retrieval

In [ ]:
wiki_safe_results = []

for q in retrieval_test_questions:
    print(f"Processing {q['id']}...")

    results = retrieve_from_wikipedia_safe(
        question_item=q,
        max_results=3,
        max_chars_per_page=1500
    )

    wiki_safe_results.extend(results)

    time.sleep(1)

print(f"Total Wikipedia Safe results collected: {len(wiki_safe_results)}")

Processing AHP_Q1...
Processing AHP_Q2...
Processing AHP_Q3...
Processing AHP_Q4...
Processing AHP_Q5...
Processing AHP_Q6...
Processing ENT_Q1...
Processing ENT_Q2...
Processing ENT_Q3...
Processing ENT_Q4...
Processing ENT_Q5...
Processing SCI_Q1...
Processing SCI_Q2...
Processing SCI_Q3...
Processing SCI_Q4...
Processing SCI_Q5...
Processing SCI_Q6...
Processing SCI_Q7...
Processing MATH_Q1...
Processing MATH_Q2...
Processing MATH_Q3...
Processing MATH_Q4...
Processing MATH_Q5...
Processing MATH_Q6...
Processing MATH_Q7...
Total Wikipedia Safe results collected: 63


### Evaluate Wikipedia safe results

In [ ]:
wiki_safe_eval_rows = []

for result in wiki_safe_results:
    q = next(item for item in retrieval_test_questions if item["id"] == result.question_id)

    eval_info = evaluate_retrieval_result(
        result=result,
        expected_answer=q["expected_answer"],
        keywords=q["keywords"]
    )

    wiki_safe_eval_rows.append({
        "retriever": result.retriever,
        "question_id": result.question_id,
        "competition": q["competition"],
        "question_type": q["question_type"],
        "retrieval_needed": q["retrieval_needed"],
        "tool_needed": q["tool_needed"],
        "rank": result.rank,
        "query": result.query,
        "title": result.title,
        "expected_answer": q["expected_answer"],
        "expected_answer_found": eval_info["expected_answer_found"],
        "keyword_hits": eval_info["keyword_hits"],
        "keyword_coverage": round(eval_info["keyword_coverage"], 3),
        "latency_seconds": round(result.latency_seconds, 3),
        "url": result.url
    })

wiki_safe_eval_df = pd.DataFrame(wiki_safe_eval_rows)

wiki_safe_eval_df

,retriever,question_id,competition,question_type,retrieval_needed,tool_needed,rank,query,title,expected_answer,expected_answer_found,keyword_hits,keyword_coverage,latency_seconds,url
0,Wikipedia Safe,AHP_Q1,Ancient History and Politics,direct_entity,True,None,1,What term describes the region centered around...,Mesopotamia,Babylonia,True,2,0.667,0.470,https://en.wikipedia.org/wiki/Mesopotamia
1,Wikipedia Safe,AHP_Q1,Ancient History and Politics,direct_entity,True,None,2,What term describes the region centered around...,History of Mesopotamia,Babylonia,False,0,0.000,0.470,https://en.wikipedia.org/wiki/History_of_Mesop...
2,Wikipedia Safe,AHP_Q1,Ancient History and Politics,direct_entity,True,None,3,What term describes the region centered around...,Babylonia,Babylonia,True,3,1.000,0.470,https://en.wikipedia.org/wiki/Babylonia
3,Wikipedia Safe,AHP_Q3,Ancient History and Politics,named_historical_figure,True,None,1,Which ancient scholar first published an editi...,Homeric Hymns,Demetrios Chalkokondyles,False,0,0.000,0.466,https://en.wikipedia.org/wiki/Homeric_Hymns
4,Wikipedia Safe,AHP_Q3,Ancient History and Politics,named_historical_figure,True,None,2,Which ancient scholar first published an editi...,Homer,Demetrios Chalkokondyles,False,1,0.333,0.466,https://en.wikipedia.org/wiki/Homer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,Wikipedia Safe,MATH_Q6,Maths,sampling_distribution,False,statistical_reasoning,2,sampling distribution mean standard deviation,Standard deviation,The shape is somewhat skewed right with a mean...,False,2,0.667,0.331,https://en.wikipedia.org/wiki/Standard_deviation
59,Wikipedia Safe,MATH_Q6,Maths,sampling_distribution,False,statistical_reasoning,3,sampling distribution mean standard deviation,Standard error,The shape is somewhat skewed right with a mean...,False,3,1.000,0.331,https://en.wikipedia.org/wiki/Standard_error
60,Wikipedia Safe,MATH_Q7,Maths,basic_arithmetic,False,calculator,1,addition arithmetic bird feed,Animal sanctuary,80,False,1,0.333,0.333,https://en.wikipedia.org/wiki/Animal_sanctuary
61,Wikipedia Safe,MATH_Q7,Maths,basic_arithmetic,False,calculator,2,addition arithmetic bird feed,Atari Falcon,80,True,1,0.333,0.333,https://en.wikipedia.org/wiki/Atari_Falcon


### Summary per question

In [ ]:
wiki_safe_eval_df["tool_needed_filled"] = wiki_safe_eval_df["tool_needed"].fillna("None")

wiki_safe_summary_df = (
    wiki_safe_eval_df
    .groupby([
        "question_id",
        "competition",
        "question_type",
        "retrieval_needed",
        "tool_needed_filled"
    ], dropna=False)
    .agg(
        total_results=("title", "count"),
        top_result_title=("title", "first"),
        top_result_expected_answer_found=("expected_answer_found", "first"),
        any_result_expected_answer_found=("expected_answer_found", "max"),
        max_keyword_coverage=("keyword_coverage", "max"),
        avg_latency_seconds=("latency_seconds", "mean")
    )
    .reset_index()
    .rename(columns={"tool_needed_filled": "tool_needed"})
)

wiki_safe_summary_df

,question_id,competition,question_type,retrieval_needed,tool_needed,total_results,top_result_title,top_result_expected_answer_found,any_result_expected_answer_found,max_keyword_coverage,avg_latency_seconds
0,AHP_Q1,Ancient History and Politics,direct_entity,True,None,3,Mesopotamia,True,True,1.000,0.470
1,AHP_Q3,Ancient History and Politics,named_historical_figure,True,None,3,Homeric Hymns,False,False,0.333,0.466
2,AHP_Q4,Ancient History and Politics,conceptual_definition,True,None,3,Sexuality in ancient Rome,False,False,0.000,0.375
3,AHP_Q5,Ancient History and Politics,comparison_reasoning,True,None,3,Ancient Greek architecture,False,False,0.000,0.404
4,AHP_Q6,Ancient History and Politics,relationship_reasoning,True,None,3,Antikythera mechanism,False,False,0.500,0.400
5,ENT_Q1,Entertainment,music_terminology,True,None,3,Ella Fitzgerald,True,True,0.667,0.422
6,ENT_Q2,Entertainment,music_voice_classification,True,None,3,Vocal range,False,True,1.000,0.451
7,ENT_Q3,Entertainment,celebrity_history,True,None,3,Death of Marilyn Monroe,False,False,0.667,0.447
8,ENT_Q4,Entertainment,media_history,True,None,3,MTV Generation,False,False,0.333,0.434
9,ENT_Q5,Entertainment,historical_motivation,True,None,3,Priscilla Presley,False,False,0.333,0.373


In [ ]:
# wiki_safe_summary_df = (
#     wiki_safe_eval_df
#     .groupby([
#         "question_id",
#         "competition",
#         "question_type",
#         "retrieval_needed",
#         "tool_needed"
#     ])
#     .agg(
#         total_results=("title", "count"),
#         top_result_title=("title", "first"),
#         top_result_expected_answer_found=("expected_answer_found", "first"),
#         any_result_expected_answer_found=("expected_answer_found", "max"),
#         max_keyword_coverage=("keyword_coverage", "max"),
#         avg_latency_seconds=("latency_seconds", "mean")
#     )
#     .reset_index()
# )

# wiki_safe_summary_df

### Overall Wikipedia benchmark

In [ ]:
total_questions = len(retrieval_test_questions)
questions_with_results = wiki_safe_summary_df["question_id"].nunique()

wiki_safe_overall_df = pd.DataFrame([{
    "retriever": "Wikipedia Safe",
    "total_questions_in_bank": total_questions,
    "questions_with_results": questions_with_results,
    "questions_without_results": total_questions - questions_with_results,
    "questions_with_expected_answer_found": int(wiki_safe_summary_df["any_result_expected_answer_found"].sum()),
    "answer_found_rate_among_returned": round(wiki_safe_summary_df["any_result_expected_answer_found"].mean(), 3),
    "answer_found_rate_overall": round(wiki_safe_summary_df["any_result_expected_answer_found"].sum() / total_questions, 3),
    "avg_max_keyword_coverage": round(wiki_safe_summary_df["max_keyword_coverage"].mean(), 3),
    "avg_latency_seconds": round(wiki_safe_summary_df["avg_latency_seconds"].mean(), 3)
}])

wiki_safe_overall_df

,retriever,total_questions_in_bank,questions_with_results,questions_without_results,questions_with_expected_answer_found,answer_found_rate_among_returned,answer_found_rate_overall,avg_max_keyword_coverage,avg_latency_seconds
0,Wikipedia Safe,25,21,4,7,0.333,0.28,0.567,0.427


### Summary by competition

In [ ]:
wiki_safe_by_competition_df = (
    wiki_safe_summary_df
    .groupby("competition")
    .agg(
        questions_with_results=("question_id", "count"),
        questions_with_expected_answer_found=("any_result_expected_answer_found", "sum"),
        avg_keyword_coverage=("max_keyword_coverage", "mean"),
        avg_latency_seconds=("avg_latency_seconds", "mean")
    )
    .reset_index()
)

wiki_safe_by_competition_df["answer_found_rate"] = (
    wiki_safe_by_competition_df["questions_with_expected_answer_found"]
    / wiki_safe_by_competition_df["questions_with_results"]
).round(3)

wiki_safe_by_competition_df

,competition,questions_with_results,questions_with_expected_answer_found,avg_keyword_coverage,avg_latency_seconds,answer_found_rate
0,Ancient History and Politics,5,1,0.366600,0.423000,0.2
1,Entertainment,5,2,0.600000,0.425400,0.4
2,Maths,5,1,0.533200,0.426600,0.2
3,Science and Nature,6,3,0.736167,0.430833,0.5


### Wikipedia Retrieval Benchmark — Findings & Conclusions

## Objective

The purpose of this experiment was to evaluate Wikipedia as a retrieval source for multiple-choice question answering across different competition categories.

The benchmark focused on:
- retrieval relevance,
- keyword coverage,
- supporting evidence presence,
- and the usefulness of retrieved context.

The experiment intentionally separated:
1. retrieval quality,
2. reasoning / answer generation,

to better understand where failures originate.

---

## Question Bank Coverage

The benchmark currently includes questions from:
- Ancient History and Politics
- Entertainment
- Science and Nature
- Maths

Each question was additionally labeled with:
- `question_type`
- `retrieval_needed`
- `tool_needed`

This enables future analysis of:
- which question types benefit from retrieval,
- which require reasoning,
- and which require external tools instead of web search.

---

## Current Evaluation Methodology

The current evaluation pipeline measures:
- whether the expected answer text appears in retrieved pages,
- keyword overlap between query and retrieved content,
- retrieval latency,
- and top-result relevance.

This benchmark currently evaluates retrieval usefulness and lexical evidence presence rather than full QA reasoning success.

---

## Important Limitation of Exact-Match Evaluation

A major finding from this experiment is that exact answer matching alone is not a reliable retrieval metric.

Examples observed:
- Relevant pages were often retrieved even when the exact answer sentence did not appear literally.
- Some irrelevant pages accidentally contained the expected answer string and were incorrectly marked as successful retrievals.

This produces:
- false negatives,
- false positives,
- and underestimates true retrieval usefulness.

---

## Main Findings

### 1. Wikipedia performs reasonably well for factual lookup questions

Strong performance was observed for:
- direct entities,
- scientific facts,
- terminology,
- chemistry formulas,
- biological concepts.

Examples:
- Atomic number → protons
- Alum formula
- Mezzo-soprano
- Root function in plants

These question types naturally align with encyclopedic retrieval.

---

### 2. Wikipedia struggles with reasoning-heavy questions

Weak performance was observed for:
- conceptual definitions,
- historical motivations,
- comparison reasoning,
- relationship reasoning.

Examples:
- Caesarism
- Doric vs Ionic comparison
- Elvis Army motivation
- Antikythera mechanism relationship reasoning

These questions require:
- synthesis,
- interpretation,
- or reasoning across retrieved evidence.

---

### 3. Maths retrieval revealed an important distinction

Several math questions produced poor retrieval behavior despite high keyword overlap.

Examples:
- arithmetic problems,
- linear algebra reasoning,
- statistical reasoning.

This demonstrates that some questions are fundamentally:

> tool-use problems rather than retrieval problems

These questions are better solved using:
- calculators,
- symbolic engines,
- mathematical reasoning systems,
- or agentic workflows.

---

## Key Insight

The experiment demonstrates that retrieval evaluation should evolve from:

> "Did the answer text appear?"

toward:

> "Did retrieval provide sufficient evidence for solving the question?"

The next stage of the benchmark will therefore shift toward:
1. comparing multiple retrieval systems,
2. evaluating semantic retrieval quality,
3. testing whether an LLM can answer correctly using retrieved context only.

---

## Next Step

The next phase will benchmark additional retrievers beyond Wikipedia, including:
- DuckDuckGo
- Tavily
- potentially Brave/Serper later

using the same:
- question bank,
- evaluation schema,
- and benchmarking methodology.

This will allow direct comparison between retrieval systems before integrating LLM-based judging and answer generation.

## DuckDuckGo

In [ ]:
!pip install duckduckgo-search -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 52.4 MB/s eta 0:00:00


In [ ]:
%%capture
!pip install ddgs -q

In [ ]:
import warnings

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

from ddgs import DDGS
import time
import pandas as pd

### DuckDuckGo retriever

In [ ]:
def retrieve_from_duckduckgo(question_item, max_results=5):
    """
    DuckDuckGo retriever using the newer ddgs package.
    """
    question_id = question_item["id"]
    query = question_item["question"]

    start_time = time.time()

    try:
        with DDGS() as ddgs:
            search_results = list(
                ddgs.text(
                    query,
                    max_results=max_results
                )
            )

    except Exception as e:
        print(f"DuckDuckGo search failed for {question_id}: {e}")
        return []

    latency = time.time() - start_time

    results = []

    for rank, item in enumerate(search_results, start=1):
        title = item.get("title", "")
        url = item.get("href", "")
        content = item.get("body", "")

        results.append(
            RetrievalResult(
                retriever="DuckDuckGo",
                question_id=question_id,
                query=query,
                title=title,
                url=url,
                content=content,
                rank=rank,
                latency_seconds=latency
            )
        )

    return results

### Run DuckDuckGo retrieval

In [ ]:
ddg_results = []

for q in retrieval_test_questions:
    print(f"Processing {q['id']}...")

    results = retrieve_from_duckduckgo(
        question_item=q,
        max_results=5
    )

    ddg_results.extend(results)

    time.sleep(1.5)

print(f"Total DuckDuckGo results collected: {len(ddg_results)}")

Processing AHP_Q1...
Processing AHP_Q2...
Processing AHP_Q3...
Processing AHP_Q4...
Processing AHP_Q5...
Processing AHP_Q6...
Processing ENT_Q1...
Processing ENT_Q2...
Processing ENT_Q3...
Processing ENT_Q4...
Processing ENT_Q5...
Processing SCI_Q1...
Processing SCI_Q2...
Processing SCI_Q3...
Processing SCI_Q4...
Processing SCI_Q5...
Processing SCI_Q6...
Processing SCI_Q7...
Processing MATH_Q1...
Processing MATH_Q2...
Processing MATH_Q3...
Processing MATH_Q4...
Processing MATH_Q5...
Processing MATH_Q6...
Processing MATH_Q7...
Total DuckDuckGo results collected: 125


### Evaluate DuckDuckGo results

In [ ]:
ddg_eval_rows = []

for result in ddg_results:
    q = next(item for item in retrieval_test_questions if item["id"] == result.question_id)

    eval_info = evaluate_retrieval_result(
        result=result,
        expected_answer=q["expected_answer"],
        keywords=q["keywords"]
    )

    ddg_eval_rows.append({
        "retriever": result.retriever,
        "question_id": result.question_id,
        "competition": q["competition"],
        "question_type": q["question_type"],
        "retrieval_needed": q["retrieval_needed"],
        "tool_needed": q["tool_needed"],
        "rank": result.rank,
        "query": result.query,
        "title": result.title,
        "expected_answer": q["expected_answer"],
        "expected_answer_found": eval_info["expected_answer_found"],
        "keyword_hits": eval_info["keyword_hits"],
        "keyword_coverage": round(eval_info["keyword_coverage"], 3),
        "latency_seconds": round(result.latency_seconds, 3),
        "url": result.url
    })

ddg_eval_df = pd.DataFrame(ddg_eval_rows)

ddg_eval_df

,retriever,question_id,competition,question_type,retrieval_needed,tool_needed,rank,query,title,expected_answer,expected_answer_found,keyword_hits,keyword_coverage,latency_seconds,url
0,DuckDuckGo,AHP_Q1,Ancient History and Politics,direct_entity,True,None,1,What term describes the region centered around...,Babylonia - Wikipedia,Babylonia,True,3,1.000,1.576,https://en.wikipedia.org/wiki/Babylonia
1,DuckDuckGo,AHP_Q1,Ancient History and Politics,direct_entity,True,None,2,What term describes the region centered around...,Mesopotamia - Wikipedia,Babylonia,True,2,0.667,1.576,https://en.wikipedia.org/wiki/Mesopotamia
2,DuckDuckGo,AHP_Q1,Ancient History and Politics,direct_entity,True,None,3,What term describes the region centered around...,Babylon - Wikipedia,Babylonia,True,2,0.667,1.576,https://en.wikipedia.org/wiki/Babylon
3,DuckDuckGo,AHP_Q1,Ancient History and Politics,direct_entity,True,None,4,What term describes the region centered around...,Geography of Mesopotamia - Wikipedia,Babylonia,False,0,0.000,1.576,https://en.wikipedia.org/wiki/Geography_of_Mes...
4,DuckDuckGo,AHP_Q1,Ancient History and Politics,direct_entity,True,None,5,What term describes the region centered around...,Sumer - Wikipedia,Babylonia,True,2,0.667,1.576,https://en.wikipedia.org/wiki/Sumer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,DuckDuckGo,MATH_Q7,Maths,basic_arithmetic,False,calculator,1,Nine bags of bird feed are in the storage room...,Yandex Disk – Cloud Storage - Apps on Google Play,80,False,0,0.000,3.012,https://play.google.com/store/apps/details/Yan...
121,DuckDuckGo,MATH_Q7,Maths,basic_arithmetic,False,calculator,2,Nine bags of bird feed are in the storage room...,Jelly Firkin Bags | firkin.world,80,False,0,0.000,3.012,https://firkin.world/
122,DuckDuckGo,MATH_Q7,Maths,basic_arithmetic,False,calculator,3,Nine bags of bird feed are in the storage room...,Feed The Birds - Mary Poppins (Julie Andrews) ...,80,False,0,0.000,3.012,https://www.youtube.com/watch?v=XHrRxQVUFN4
123,DuckDuckGo,MATH_Q7,Maths,basic_arithmetic,False,calculator,4,Nine bags of bird feed are in the storage room...,[Bug] Agent terminated due to error You can pr...,80,False,0,0.000,3.012,https://github.com/Draculabo/AntigravityManage...


### Summary per question

In [ ]:
ddg_eval_df["tool_needed_filled"] = ddg_eval_df["tool_needed"].fillna("None")

ddg_summary_df = (
    ddg_eval_df
    .groupby([
        "question_id",
        "competition",
        "question_type",
        "retrieval_needed",
        "tool_needed_filled"
    ], dropna=False)
    .agg(
        total_results=("title", "count"),
        top_result_title=("title", "first"),
        top_result_expected_answer_found=("expected_answer_found", "first"),
        any_result_expected_answer_found=("expected_answer_found", "max"),
        max_keyword_coverage=("keyword_coverage", "max"),
        avg_latency_seconds=("latency_seconds", "mean")
    )
    .reset_index()
    .rename(columns={"tool_needed_filled": "tool_needed"})
)

ddg_summary_df

,question_id,competition,question_type,retrieval_needed,tool_needed,total_results,top_result_title,top_result_expected_answer_found,any_result_expected_answer_found,max_keyword_coverage,avg_latency_seconds
0,AHP_Q1,Ancient History and Politics,direct_entity,True,None,5,Babylonia - Wikipedia,True,True,1.000,1.576
1,AHP_Q2,Ancient History and Politics,artifact_material,True,None,5,Roman dodecahedron - Wikipedia,True,True,0.500,1.661
2,AHP_Q3,Ancient History and Politics,named_historical_figure,True,None,5,Homer - Wikipedia,True,True,0.667,2.026
3,AHP_Q4,Ancient History and Politics,conceptual_definition,True,None,5,Caesarism - Wikipedia,False,False,1.000,1.301
4,AHP_Q5,Ancient History and Politics,comparison_reasoning,True,None,5,Classical order - Wikipedia,False,False,0.800,1.052
5,AHP_Q6,Ancient History and Politics,relationship_reasoning,True,None,5,"(PDF) The Antikythera Mechanism, Rhodes, and E...",False,False,0.500,1.615
6,ENT_Q1,Entertainment,music_terminology,True,None,5,Scat singing - Wikipedia,True,True,0.667,1.081
7,ENT_Q2,Entertainment,music_voice_classification,True,None,5,Voice type - Wikipedia,False,True,0.667,1.619
8,ENT_Q3,Entertainment,celebrity_history,True,None,5,Death of Marilyn Monroe - Wikipedia,False,False,0.667,0.972
9,ENT_Q4,Entertainment,media_history,True,None,5,MTV - Wikipedia,False,False,0.667,1.124


### Overall benchmark

In [ ]:
total_questions = len(retrieval_test_questions)
questions_with_results = ddg_summary_df["question_id"].nunique()

ddg_overall_df = pd.DataFrame([{
    "retriever": "DuckDuckGo",
    "total_questions_in_bank": total_questions,
    "questions_with_results": questions_with_results,
    "questions_without_results": total_questions - questions_with_results,
    "questions_with_expected_answer_found": int(ddg_summary_df["any_result_expected_answer_found"].sum()),
    "answer_found_rate_among_returned": round(ddg_summary_df["any_result_expected_answer_found"].mean(), 3),
    "answer_found_rate_overall": round(ddg_summary_df["any_result_expected_answer_found"].sum() / total_questions, 3),
    "avg_max_keyword_coverage": round(ddg_summary_df["max_keyword_coverage"].mean(), 3),
    "avg_latency_seconds": round(ddg_summary_df["avg_latency_seconds"].mean(), 3)
}])

ddg_overall_df

,retriever,total_questions_in_bank,questions_with_results,questions_without_results,questions_with_expected_answer_found,answer_found_rate_among_returned,answer_found_rate_overall,avg_max_keyword_coverage,avg_latency_seconds
0,DuckDuckGo,25,25,0,9,0.36,0.36,0.685,1.646


### Summary by competition

In [ ]:
ddg_by_competition_df = (
    ddg_summary_df
    .groupby("competition")
    .agg(
        questions_with_results=("question_id", "count"),
        questions_with_expected_answer_found=("any_result_expected_answer_found", "sum"),
        avg_keyword_coverage=("max_keyword_coverage", "mean"),
        avg_latency_seconds=("avg_latency_seconds", "mean")
    )
    .reset_index()
)

ddg_by_competition_df["answer_found_rate"] = (
    ddg_by_competition_df["questions_with_expected_answer_found"]
    / ddg_by_competition_df["questions_with_results"]
).round(3)

ddg_by_competition_df

,competition,questions_with_results,questions_with_expected_answer_found,avg_keyword_coverage,avg_latency_seconds,answer_found_rate
0,Ancient History and Politics,6,3,0.744500,1.538500,0.500
1,Entertainment,5,2,0.733600,1.161000,0.400
2,Maths,7,0,0.476286,2.100143,0.000
3,Science and Nature,7,4,0.809714,1.630000,0.571


### DuckDuckGo Retriever Evaluation Summary

The DuckDuckGo retriever demonstrated noticeably stronger retrieval performance compared to the initial Wikipedia-only retriever baseline.

### Key Improvements

- Overall answer retrieval rate improved from **0.28 → 0.32**
- Average keyword coverage improved from **0.567 → 0.697**
- All benchmark questions returned results successfully
- Retrieval diversity improved significantly due to access to broader web sources instead of only Wikipedia titles/pages

### Important Observations

The DuckDuckGo experiment revealed an important limitation in the current evaluation methodology.

Several questions produced highly relevant retrieved documents with strong keyword coverage, but were still marked as failures because the evaluator currently relies on:
- exact expected answer string matching
- lexical overlap

rather than:
- semantic relevance
- retrieval sufficiency
- reasoning capability

This became especially visible in:
- mathematical reasoning questions
- statistical reasoning questions
- calculator-style problems

where retrieval quality may actually be good, but answer extraction requires additional reasoning or computation.

### Main Insight

The current benchmark is still mixing together:
1. Retrieval failures
2. Reasoning failures
3. Evaluation methodology limitations

The DuckDuckGo results indicate that retrieval quality is likely stronger than the current benchmark metrics suggest.

### Conclusion

The DuckDuckGo experiment validated that:
- broader web retrieval improves recall and coverage
- retrieval benchmarking infrastructure is functioning correctly
- query formulation quality strongly affects retrieval quality
- future evaluation should move toward semantic or LLM-based judging rather than exact answer matching

This concludes the DuckDuckGo retrieval benchmark phase.

## Tavili

In [ ]:
%%capture
!pip install -U langchain-tavily

In [ ]:
import os
import time
import pandas as pd
from langchain_tavily import TavilySearch
from google.colab import userdata

os.environ["TAVILY_API_KEY"] = userdata.get("Tavili-API-Key")

tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general"
)

print("Tavily search tool initialized")

Tavily search tool initialized


In [ ]:
results = tavily_search_tool.invoke({
    "query": "Roman dodecahedra purpose"
})

results

{'query': 'Roman dodecahedra purpose',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.reddit.com/r/AlternativeHistory/comments/10jnmc7/roman_dodecahedron_anyone_have_solid_ideas_on_the/',
   'title': 'Roman dodecahedron, anyone have solid ideas on the purpose?',
   'content': "It's clearly used to measure or calibrate somthing of standard but varying sizes. Think, ring makers, they allow you to precisely gauge the ring",
   'score': 0.7760977,
   'raw_content': None},
  {'url': 'https://en.wikipedia.org/wiki/Roman_dodecahedron',
   'title': 'Roman dodecahedron - Wikipedia',
   'content': 'Their purpose or meaning has been long debated but remains unknown.',
   'score': 0.62330866,
   'raw_content': None},
  {'url': 'https://www.livescience.com/archaeology/romans/roman-dodecahedron-a-mysterious-12-sided-object-that-has-baffled-archaeologists-for-centuries',
   'title': 'Roman dodecahedron: A mysterious 12-sided object that has ...',
   'c

### Tavily retriever

In [ ]:
def retrieve_from_tavily(question_item, max_results=5):
    question_id = question_item["id"]
    query = question_item["question"]

    start_time = time.time()

    try:
        response = tavily_search_tool.invoke({
        "query": query
        })


    except Exception as e:
        print(f"Tavily search failed for {question_id}: {e}")
        return []

    latency = time.time() - start_time

    search_results = response.get("results", [])

    results = []

    for rank, item in enumerate(search_results, start=1):
        title = item.get("title", "")
        url = item.get("url", "")
        content = item.get("content", "")

        results.append(
            RetrievalResult(
                retriever="Tavily",
                question_id=question_id,
                query=query,
                title=title,
                url=url,
                content=content,
                rank=rank,
                latency_seconds=latency
            )
        )

    return results

### Run Tavily retrieval

In [ ]:
tavily_results = []

for q in retrieval_test_questions:
    print(f"Processing {q['id']}...")

    results = retrieve_from_tavily(
        question_item=q,
        max_results=5
    )

    tavily_results.extend(results)

    time.sleep(1.5)

print(f"Total Tavily results collected: {len(tavily_results)}")

Processing AHP_Q1...
Processing AHP_Q2...
Processing AHP_Q3...
Processing AHP_Q4...
Processing AHP_Q5...
Processing AHP_Q6...
Processing ENT_Q1...
Processing ENT_Q2...
Processing ENT_Q3...
Processing ENT_Q4...
Processing ENT_Q5...
Processing SCI_Q1...
Processing SCI_Q2...
Processing SCI_Q3...
Processing SCI_Q4...
Processing SCI_Q5...
Processing SCI_Q6...
Processing SCI_Q7...
Processing MATH_Q1...
Processing MATH_Q2...
Processing MATH_Q3...
Processing MATH_Q4...
Processing MATH_Q5...
Processing MATH_Q6...
Processing MATH_Q7...
Total Tavily results collected: 125


### Evaluate Tavily results

In [ ]:
tavily_eval_rows = []

for result in tavily_results:
    q = next(item for item in retrieval_test_questions if item["id"] == result.question_id)

    eval_info = evaluate_retrieval_result(
        result=result,
        expected_answer=q["expected_answer"],
        keywords=q["keywords"]
    )

    tavily_eval_rows.append({
        "retriever": result.retriever,
        "question_id": result.question_id,
        "competition": q["competition"],
        "question_type": q["question_type"],
        "retrieval_needed": q["retrieval_needed"],
        "tool_needed": q["tool_needed"],
        "rank": result.rank,
        "query": result.query,
        "title": result.title,
        "expected_answer": q["expected_answer"],
        "expected_answer_found": eval_info["expected_answer_found"],
        "keyword_hits": eval_info["keyword_hits"],
        "keyword_coverage": round(eval_info["keyword_coverage"], 3),
        "latency_seconds": round(result.latency_seconds, 3),
        "url": result.url
    })

tavily_eval_df = pd.DataFrame(tavily_eval_rows)

tavily_eval_df

,retriever,question_id,competition,question_type,retrieval_needed,tool_needed,rank,query,title,expected_answer,expected_answer_found,keyword_hits,keyword_coverage,latency_seconds,url
0,Tavily,AHP_Q1,Ancient History and Politics,direct_entity,True,None,1,What term describes the region centered around...,Babylonia - Wikipedia,Babylonia,True,3,1.000,0.113,https://en.wikipedia.org/wiki/Babylonia
1,Tavily,AHP_Q1,Ancient History and Politics,direct_entity,True,None,2,What term describes the region centered around...,[FREE] Define these terms: - City-state: An in...,Babylonia,True,3,1.000,0.113,https://brainly.com/question/28567584
2,Tavily,AHP_Q1,Ancient History and Politics,direct_entity,True,None,3,What term describes the region centered around...,"Babylonia | History, Map, Culture, & Facts | B...",Babylonia,True,2,0.667,0.113,https://www.britannica.com/place/Babylonia
3,Tavily,AHP_Q1,Ancient History and Politics,direct_entity,True,None,4,What term describes the region centered around...,Babylon and the cities and tribes of Southern ...,Babylonia,True,2,0.667,0.113,https://oracc.museum.upenn.edu/saao/aebp/Essen...
4,Tavily,AHP_Q1,Ancient History and Politics,direct_entity,True,None,5,What term describes the region centered around...,When it comes to the historically rich region ...,Babylonia,True,2,0.667,0.113,https://www.facebook.com/groups/Archaeology.Pr...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,Tavily,MATH_Q7,Maths,basic_arithmetic,False,calculator,1,Nine bags of bird feed are in the storage room...,Nine bags of bird feed are in the storage room...,80,False,1,0.333,2.429,https://www.gauthmath.com/solution/18134085872...
121,Tavily,MATH_Q7,Maths,basic_arithmetic,False,calculator,2,Nine bags of bird feed are in the storage room...,Word problem | Free Math Help Forum,80,False,0,0.000,2.429,https://www.freemathhelp.com/forum/threads/wor...
122,Tavily,MATH_Q7,Maths,basic_arithmetic,False,calculator,3,Nine bags of bird feed are in the storage room...,ASVAB Arithmetic Reasoning Subtest Sample Ques...,80,False,1,0.333,2.429,https://www.dummies.com/article/academics-the-...
123,Tavily,MATH_Q7,Maths,basic_arithmetic,False,calculator,4,Nine bags of bird feed are in the storage room...,Question 17 - Algebra 2 - TNReady Practice Tes...,80,False,0,0.000,2.429,https://www.youtube.com/watch?v=Lc3VFFMUph4


### Tavily summary per question

In [ ]:
tavily_eval_df["tool_needed_filled"] = tavily_eval_df["tool_needed"].fillna("None")

tavily_summary_df = (
    tavily_eval_df
    .groupby([
        "question_id",
        "competition",
        "question_type",
        "retrieval_needed",
        "tool_needed_filled"
    ], dropna=False)
    .agg(
        total_results=("title", "count"),
        top_result_title=("title", "first"),
        top_result_expected_answer_found=("expected_answer_found", "first"),
        any_result_expected_answer_found=("expected_answer_found", "max"),
        max_keyword_coverage=("keyword_coverage", "max"),
        avg_latency_seconds=("latency_seconds", "mean")
    )
    .reset_index()
    .rename(columns={"tool_needed_filled": "tool_needed"})
)

tavily_summary_df

,question_id,competition,question_type,retrieval_needed,tool_needed,total_results,top_result_title,top_result_expected_answer_found,any_result_expected_answer_found,max_keyword_coverage,avg_latency_seconds
0,AHP_Q1,Ancient History and Politics,direct_entity,True,None,5,Babylonia - Wikipedia,True,True,1.000,0.113
1,AHP_Q2,Ancient History and Politics,artifact_material,True,None,5,Roman dodecahedron - Wikipedia,True,True,0.750,0.801
2,AHP_Q3,Ancient History and Politics,named_historical_figure,True,None,5,Homer - Wikipedia,True,True,1.000,0.949
3,AHP_Q4,Ancient History and Politics,conceptual_definition,True,None,5,Caesarism - Wikipedia,False,False,0.333,0.909
4,AHP_Q5,Ancient History and Politics,comparison_reasoning,True,None,5,Doric Ionic and Corinthian Columns: The Pillar...,False,False,0.600,0.784
5,AHP_Q6,Ancient History and Politics,relationship_reasoning,True,None,5,What is the origin of the Antikythera mechanis...,False,False,0.500,0.846
6,ENT_Q1,Entertainment,music_terminology,True,None,5,WHAT Definition & Meaning - Dictionary.com,False,False,0.000,1.037
7,ENT_Q2,Entertainment,music_voice_classification,True,None,5,Voice type - Wikipedia,True,True,0.667,0.738
8,ENT_Q3,Entertainment,celebrity_history,True,None,5,How Did Marilyn Monroe Die? - History News Net...,False,False,1.000,0.923
9,ENT_Q4,Entertainment,media_history,True,None,5,How did the rise of MTV in the 1980s change th...,False,False,0.667,0.909


### Overall Tavily benchmark

In [ ]:
total_questions = len(retrieval_test_questions)
questions_with_results = tavily_summary_df["question_id"].nunique()

tavily_overall_df = pd.DataFrame([{
    "retriever": "Tavily",
    "total_questions_in_bank": total_questions,
    "questions_with_results": questions_with_results,
    "questions_without_results": total_questions - questions_with_results,
    "questions_with_expected_answer_found": int(tavily_summary_df["any_result_expected_answer_found"].sum()),
    "answer_found_rate_among_returned": round(tavily_summary_df["any_result_expected_answer_found"].mean(), 3),
    "answer_found_rate_overall": round(tavily_summary_df["any_result_expected_answer_found"].sum() / total_questions, 3),
    "avg_max_keyword_coverage": round(tavily_summary_df["max_keyword_coverage"].mean(), 3),
    "avg_latency_seconds": round(tavily_summary_df["avg_latency_seconds"].mean(), 3)
}])

tavily_overall_df

,retriever,total_questions_in_bank,questions_with_results,questions_without_results,questions_with_expected_answer_found,answer_found_rate_among_returned,answer_found_rate_overall,avg_max_keyword_coverage,avg_latency_seconds
0,Tavily,25,25,0,7,0.28,0.28,0.664,1.18


### Tavily by competition

In [ ]:
tavily_by_competition_df = (
    tavily_summary_df
    .groupby("competition")
    .agg(
        questions_with_results=("question_id", "count"),
        questions_with_expected_answer_found=("any_result_expected_answer_found", "sum"),
        avg_keyword_coverage=("max_keyword_coverage", "mean"),
        avg_latency_seconds=("avg_latency_seconds", "mean")
    )
    .reset_index()
)

tavily_by_competition_df["answer_found_rate"] = (
    tavily_by_competition_df["questions_with_expected_answer_found"]
    / tavily_by_competition_df["questions_with_results"]
).round(3)

tavily_by_competition_df

,competition,questions_with_results,questions_with_expected_answer_found,avg_keyword_coverage,avg_latency_seconds,answer_found_rate
0,Ancient History and Politics,6,3,0.697167,0.733667,0.500
1,Entertainment,5,1,0.666800,0.897800,0.200
2,Maths,7,0,0.571429,1.713000,0.000
3,Science and Nature,7,3,0.726286,1.231286,0.429


### Tavily Retrieval Evaluation Summary

We evaluated Tavily using the same benchmark and evaluation methodology applied previously to Wikipedia Safe and DuckDuckGo retrieval.

### Key Observations

- Tavily successfully returned results for all 25 benchmark questions.
- Overall answer retrieval performance reached:
  - **28% answer found rate overall**
  - **66.4% average keyword coverage**
  - **~1.18s average latency**

### Domain-Level Insights

| Competition | Answer Found Rate | Avg Keyword Coverage |
|---|---|---|
| Ancient History & Politics | 50.0% | 0.697 |
| Entertainment | 20.0% | 0.667 |
| Maths | 0.0% | 0.571 |
| Science & Nature | 42.9% | 0.726 |

### Important Findings

- Tavily performed strongly on:
  - direct factual lookup
  - named entities
  - historical references
  - science-related retrieval

- Tavily still struggled with:
  - reasoning-heavy questions
  - conceptual interpretation
  - mathematical solving
  - symbolic/statistical reasoning

- Compared to DuckDuckGo:
  - Tavily produced more semantically relevant results
  - Fewer completely irrelevant retrieval failures occurred
  - Result quality was generally more focused and contextual

### Critical Observation About Evaluation

The current evaluation still depends primarily on:
- expected-answer keyword matching
- keyword coverage overlap

This remains useful for benchmarking retrieval consistency, but it is insufficient for measuring true retrieval usefulness in reasoning-oriented questions.

Examples:
- A retrieved document may contain highly useful context without explicitly containing the expected answer string.
- Some retrievals are semantically correct but fail keyword-based evaluation.
- Maths and reasoning questions especially expose this limitation.

### Conclusion

Tavily demonstrated better semantic retrieval behavior than DuckDuckGo, especially for factual and contextual knowledge retrieval.

However, the benchmark also confirmed that:
- retrieval quality alone is not enough,
- and keyword matching is not a sufficient evaluator for semantic usefulness.

The next phase should therefore focus on:
1. expanding retrieval-source benchmarking,
2. improving retrieval query generation,
3. and integrating LLM-based judging/evaluation for semantic relevance and usefulness.

Mathematical and reasoning-heavy tasks will likely require a separate agentic or tool-augmented evaluation pipeline rather than standard retrieval evaluation alone.

In [ ]:
# Single-question experiment:
# Testing whether Tavily accepts competition names as topic values.

from langchain_tavily import TavilySearch

# Pick one question
q = retrieval_test_questions[0]

competition_topic = q["competition"]

print("Question ID:", q["id"])
print("Competition:", competition_topic)
print("Question:", q["question"])

print("\n" + "="*80)
print(f"Testing topic = '{competition_topic}'")

try:
    tavily_tool = TavilySearch(
        max_results=5,
        topic=competition_topic
    )

    response = tavily_tool.invoke({
        "query": q["question"]
    })

    results = response.get("results", [])

    print(f"Results returned: {len(results)}")

    for idx, item in enumerate(results[:3], start=1):
        print(f"\nResult #{idx}")
        print("Title:", item.get("title", ""))
        print("URL:", item.get("url", ""))
        print("Content snippet:", item.get("content", "")[:300])

except Exception as e:
    print("Error:", e)

Question ID: AHP_Q1
Competition: Ancient History and Politics
Question: What term describes the region centered around the city of Babylon in central-southern Mesopotamia?

Testing topic = 'Ancient History and Politics'
Error: 1 validation error for TavilySearch
topic
  Input should be 'general', 'news' or 'finance' [type=literal_error, input_value='Ancient History and Politics', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/literal_error


## SerpAPI

In [ ]:
# Install SerpAPI client

%%capture
!pip install google-search-results

In [ ]:
import os
import time
from serpapi import GoogleSearch

# Load API key
SERPAPI_API_KEY = userdata.get("SerpAPI")

os.environ["SERPAPI_API_KEY"] = SERPAPI_API_KEY

print("SerpAPI initialized")

SerpAPI initialized


### SerAPI Retriever

In [ ]:
from serpapi import GoogleSearch
import time
import pandas as pd


def retrieve_from_serpapi(question_item, max_results=5):
    """
    SerpAPI Google Search retriever.

    Uses Google organic search results as raw retrieval evidence.
    """
    question_id = question_item["id"]
    query = question_item["question"]

    params = {
        "engine": "google",
        "q": query,
        "api_key": SERPAPI_API_KEY,
        "num": max_results
    }

    start_time = time.time()

    try:
        search = GoogleSearch(params)
        response = search.get_dict()

    except Exception as e:
        print(f"SerpAPI search failed for {question_id}: {e}")
        return []

    latency = time.time() - start_time

    organic_results = response.get("organic_results", [])

    results = []

    for rank, item in enumerate(organic_results[:max_results], start=1):
        title = item.get("title", "")
        url = item.get("link", "")
        content = item.get("snippet", "")

        results.append(
            RetrievalResult(
                retriever="SerpAPI Google",
                question_id=question_id,
                query=query,
                title=title,
                url=url,
                content=content,
                rank=rank,
                latency_seconds=latency
            )
        )

    return results

### Run SerpAPI benchmark

In [ ]:
serpapi_results = []

for q in retrieval_test_questions:
    print(f"Processing {q['id']}...")

    results = retrieve_from_serpapi(
        question_item=q,
        max_results=5
    )

    serpapi_results.extend(results)

    time.sleep(1.5)

print(f"Total SerpAPI results collected: {len(serpapi_results)}")


Processing AHP_Q1...
Processing AHP_Q2...
Processing AHP_Q3...
Processing AHP_Q4...
Processing AHP_Q5...
Processing AHP_Q6...
Processing ENT_Q1...
Processing ENT_Q2...
Processing ENT_Q3...
Processing ENT_Q4...
Processing ENT_Q5...
Processing SCI_Q1...
Processing SCI_Q2...
Processing SCI_Q3...
Processing SCI_Q4...
Processing SCI_Q5...
Processing SCI_Q6...
Processing SCI_Q7...
Processing MATH_Q1...
Processing MATH_Q2...
Processing MATH_Q3...
Processing MATH_Q4...
Processing MATH_Q5...
Processing MATH_Q6...
Processing MATH_Q7...
Total SerpAPI results collected: 125


### Evaluate SerpAPI results

In [ ]:
serpapi_eval_rows = []

for result in serpapi_results:
    q = next(item for item in retrieval_test_questions if item["id"] == result.question_id)

    eval_info = evaluate_retrieval_result(
        result=result,
        expected_answer=q["expected_answer"],
        keywords=q["keywords"]
    )

    serpapi_eval_rows.append({
        "retriever": result.retriever,
        "question_id": result.question_id,
        "competition": q["competition"],
        "question_type": q["question_type"],
        "retrieval_needed": q["retrieval_needed"],
        "tool_needed": q["tool_needed"],
        "rank": result.rank,
        "query": result.query,
        "title": result.title,
        "expected_answer": q["expected_answer"],
        "expected_answer_found": eval_info["expected_answer_found"],
        "keyword_hits": eval_info["keyword_hits"],
        "keyword_coverage": round(eval_info["keyword_coverage"], 3),
        "latency_seconds": round(result.latency_seconds, 3),
        "url": result.url
    })

serpapi_eval_df = pd.DataFrame(serpapi_eval_rows)

serpapi_eval_df

,retriever,question_id,competition,question_type,retrieval_needed,tool_needed,rank,query,title,expected_answer,expected_answer_found,keyword_hits,keyword_coverage,latency_seconds,url
0,SerpAPI Google,AHP_Q1,Ancient History and Politics,direct_entity,True,None,1,What term describes the region centered around...,Babylonia,Babylonia,True,3,1.000,0.092,https://en.wikipedia.org/wiki/Babylonia
1,SerpAPI Google,AHP_Q1,Ancient History and Politics,direct_entity,True,None,2,What term describes the region centered around...,"Babylonia | History, Map, Culture, & Facts",Babylonia,True,2,0.667,0.092,https://www.britannica.com/place/Babylonia
2,SerpAPI Google,AHP_Q1,Ancient History and Politics,direct_entity,True,None,3,What term describes the region centered around...,Flashcards Vocab quiz let,Babylonia,False,1,0.333,0.092,https://quizlet.com/1097282325/flashcards?funn...
3,SerpAPI Google,AHP_Q1,Ancient History and Politics,direct_entity,True,None,4,What term describes the region centered around...,Mesopotamia,Babylonia,False,0,0.000,0.092,https://en.wikipedia.org/wiki/Mesopotamia
4,SerpAPI Google,AHP_Q1,Ancient History and Politics,direct_entity,True,None,5,What term describes the region centered around...,"I hear people use the names Babylon, Assyria, ...",Babylonia,False,1,0.333,0.092,https://www.reddit.com/r/AskHistorians/comment...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,SerpAPI Google,MATH_Q7,Maths,basic_arithmetic,False,calculator,1,Nine bags of bird feed are in the storage room...,Nine bags of bird feed are in the storage room...,80,False,1,0.333,6.303,https://www.gauthmath.com/solution/18134085872...
121,SerpAPI Google,MATH_Q7,Maths,basic_arithmetic,False,calculator,2,Nine bags of bird feed are in the storage room...,"9 Boxes, only use #'s 1 through 9 to Solve | M...",80,False,0,0.000,6.303,https://www.youtube.com/watch?v=xTMxF7uFFmE
122,SerpAPI Google,MATH_Q7,Maths,basic_arithmetic,False,calculator,3,Nine bags of bird feed are in the storage room...,Math teas (38q) Flashcards,80,False,0,0.000,6.303,https://quizlet.com/971274832/math-teas-38q-fl...
123,SerpAPI Google,MATH_Q7,Maths,basic_arithmetic,False,calculator,4,Nine bags of bird feed are in the storage room...,Question 17 - Algebra 2 - TNReady Practice Test,80,False,0,0.000,6.303,https://www.youtube.com/watch?v=Lc3VFFMUph4


### SerpAPI summary per question

In [ ]:
serpapi_eval_df["tool_needed_filled"] = serpapi_eval_df["tool_needed"].fillna("None")

serpapi_summary_df = (
    serpapi_eval_df
    .groupby([
        "question_id",
        "competition",
        "question_type",
        "retrieval_needed",
        "tool_needed_filled"
    ], dropna=False)
    .agg(
        total_results=("title", "count"),
        top_result_title=("title", "first"),
        top_result_expected_answer_found=("expected_answer_found", "first"),
        any_result_expected_answer_found=("expected_answer_found", "max"),
        max_keyword_coverage=("keyword_coverage", "max"),
        avg_latency_seconds=("latency_seconds", "mean")
    )
    .reset_index()
    .rename(columns={"tool_needed_filled": "tool_needed"})
)

serpapi_summary_df

,question_id,competition,question_type,retrieval_needed,tool_needed,total_results,top_result_title,top_result_expected_answer_found,any_result_expected_answer_found,max_keyword_coverage,avg_latency_seconds
0,AHP_Q1,Ancient History and Politics,direct_entity,True,None,5,Babylonia,True,True,1.000,0.092
1,AHP_Q2,Ancient History and Politics,artifact_material,True,None,5,Roman dodecahedron,True,True,0.750,1.794
2,AHP_Q3,Ancient History and Politics,named_historical_figure,True,None,5,Homeric Hymns,True,True,1.000,2.414
3,AHP_Q4,Ancient History and Politics,conceptual_definition,True,None,5,Caesarism,False,False,0.333,1.887
4,AHP_Q5,Ancient History and Politics,comparison_reasoning,True,None,5,The 3 Orders of Ancient Greek Architecture,False,False,0.600,1.391
5,AHP_Q6,Ancient History and Politics,relationship_reasoning,True,None,5,Antikythera mechanism,False,False,0.500,0.964
6,ENT_Q1,Entertainment,music_terminology,True,None,5,Watch Jazzmeia Horn Explain The Legacy Of Ella...,False,True,0.667,0.890
7,ENT_Q2,Entertainment,music_voice_classification,True,None,5,Vocal Ranges | Yale University Library,True,True,0.667,1.513
8,ENT_Q3,Entertainment,celebrity_history,True,None,5,Death of Marilyn Monroe,False,False,0.667,1.895
9,ENT_Q4,Entertainment,media_history,True,None,5,The Impact of MTV in the 1980s,False,False,0.333,0.807


### Overall SerpAPI benchmark

In [ ]:
total_questions = len(retrieval_test_questions)
questions_with_results = serpapi_summary_df["question_id"].nunique()

serpapi_overall_df = pd.DataFrame([{
    "retriever": "SerpAPI Google",
    "total_questions_in_bank": total_questions,
    "questions_with_results": questions_with_results,
    "questions_without_results": total_questions - questions_with_results,
    "questions_with_expected_answer_found": int(serpapi_summary_df["any_result_expected_answer_found"].sum()),
    "answer_found_rate_among_returned": round(serpapi_summary_df["any_result_expected_answer_found"].mean(), 3),
    "answer_found_rate_overall": round(serpapi_summary_df["any_result_expected_answer_found"].sum() / total_questions, 3),
    "avg_max_keyword_coverage": round(serpapi_summary_df["max_keyword_coverage"].mean(), 3),
    "avg_latency_seconds": round(serpapi_summary_df["avg_latency_seconds"].mean(), 3)
}])

serpapi_overall_df

,retriever,total_questions_in_bank,questions_with_results,questions_without_results,questions_with_expected_answer_found,answer_found_rate_among_returned,answer_found_rate_overall,avg_max_keyword_coverage,avg_latency_seconds
0,SerpAPI Google,25,25,0,9,0.36,0.36,0.671,2.645


### SerpAPI by competition

In [ ]:
serpapi_by_competition_df = (
    serpapi_summary_df
    .groupby("competition")
    .agg(
        questions_with_results=("question_id", "count"),
        questions_with_expected_answer_found=("any_result_expected_answer_found", "sum"),
        avg_keyword_coverage=("max_keyword_coverage", "mean"),
        avg_latency_seconds=("avg_latency_seconds", "mean")
    )
    .reset_index()
)

serpapi_by_competition_df["answer_found_rate"] = (
    serpapi_by_competition_df["questions_with_expected_answer_found"]
    / serpapi_by_competition_df["questions_with_results"]
).round(3)

serpapi_by_competition_df

,competition,questions_with_results,questions_with_expected_answer_found,avg_keyword_coverage,avg_latency_seconds,answer_found_rate
0,Ancient History and Politics,6,3,0.697167,1.423667,0.500
1,Entertainment,5,2,0.666800,1.390000,0.400
2,Maths,7,1,0.595286,3.576429,0.143
3,Science and Nature,7,3,0.726429,3.656000,0.429


### SerpAPI (Google Search) Retrieval Evaluation Summary

We evaluated SerpAPI Google Search using the same benchmark framework and evaluation methodology applied previously to:
- Wikipedia Safe
- DuckDuckGo
- Tavily

### Overall Benchmark Results

| Retriever | Answer Found Rate | Avg Keyword Coverage | Avg Latency |
|---|---|---|---|
| Wikipedia Safe | 28.0% | 0.567 | 0.03s |
| DuckDuckGo | 32.0% | 0.697 | 1.58s |
| Tavily | 28.0% | 0.664 | 1.18s |
| **SerpAPI Google** | **36.0%** | **0.671** | **2.85s** |

### Domain-Level Results

| Competition | Answer Found Rate | Avg Keyword Coverage |
|---|---|---|
| Ancient History & Politics | 50.0% | 0.697 |
| Entertainment | 40.0% | 0.667 |
| Maths | 14.3% | 0.595 |
| Science & Nature | 42.9% | 0.726 |

### Key Observations

- SerpAPI produced the strongest overall retrieval performance among all tested retrievers.
- Google ranking quality appears significantly stronger for:
  - factual lookup
  - historical entities
  - scientific definitions
  - contextual knowledge retrieval

- Retrieval quality improvements were especially noticeable in:
  - top-ranked result relevance
  - semantic alignment between query and result title
  - reduction of irrelevant or noisy results

### Important Limitation Identified

Although SerpAPI improved retrieval quality, the benchmark exposed a growing limitation in the current evaluation methodology.

The current evaluator primarily depends on:
- expected-answer keyword matching
- keyword overlap coverage

This approach increasingly fails to capture:
- semantic usefulness
- contextual relevance
- reasoning support quality

Several retrievals returned highly useful context while still being marked as failures because the exact expected answer string was not explicitly present.

This issue became especially visible for:
- reasoning-heavy questions
- conceptual interpretation
- mathematics/statistics problems

### Final Conclusion of Retrieval Benchmarking Phase

The retrieval experiments confirmed that:
- retrieval engine choice significantly impacts benchmark quality,
- Google-based retrieval currently performs best among tested systems,
- but retrieval evaluation itself now requires semantic understanding rather than keyword-only heuristics.

The next phase should therefore focus on:
1. query engineering and reformulation experiments,
2. semantic/LLM-based retrieval judging,
3. hybrid retrieval + reasoning evaluation,
4. and eventually agentic handling for mathematical and reasoning-intensive tasks.

At this stage, the benchmark framework is already capable of producing meaningful comparative retrieval analysis across multiple retrieval engines.

## Query Strategy Benchmarking



In the previous retrieval benchmarks, we used mostly the original question as the search query.

However, search engines may behave differently depending on how the query is written. A full natural-language question may contain filler words or misleading words, while a condensed keyword query may retrieve more focused results.

In this section, we test different query construction strategies using free/low-cost retrievers first:

- Wikipedia
- DuckDuckGo

The goal is to identify which query strategies improve retrieval quality before spending limited Tavily or SerpAPI credits.

We compare the following query strategies:

1. `raw_question` — original question text
2. `keyword_condensed` — important keywords only
3. `domain_prefixed` — competition/domain added before the question
4. `entity_focused` — compact entity/concept-focused query
5. `expected_answer_oracle` — expected answer added to the query, used only as an upper-bound diagnostic, not for real gameplay

### Query strategy functions

In [ ]:
import re

STOPWORDS = {
    "what", "which", "who", "when", "where", "why", "how",
    "is", "are", "was", "were", "be", "been", "being",
    "do", "does", "did",
    "the", "a", "an",
    "of", "in", "on", "at", "to", "for", "from", "with", "by", "as",
    "and", "or", "but",
    "this", "that", "these", "those",
    "following", "primary", "main", "general", "fundamental",
    "purpose", "term", "type", "part"
}

DOMAIN_PREFIXES = {
    "Ancient History and Politics": "ancient history politics",
    "Entertainment": "entertainment music film celebrity media",
    "Science and Nature": "science nature biology chemistry physics astronomy",
    "Maths": "mathematics statistics algebra geometry"
}


def normalize_text_for_query(text):
    text = text.replace("·", " ")
    text = re.sub(r"[^a-zA-Z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def build_keyword_condensed_query(question_item, max_words=10):
    question = normalize_text_for_query(question_item["question"])
    words = question.split()

    filtered_words = [
        w for w in words
        if w.lower() not in STOPWORDS and len(w) > 2
    ]

    return " ".join(filtered_words[:max_words])


def build_domain_prefixed_query(question_item):
    domain_prefix = DOMAIN_PREFIXES.get(question_item["competition"], "")
    return f"{domain_prefix} {question_item['question']}"


def build_entity_focused_query(question_item):
    """
    Simple entity-focused strategy using provided keywords.
    This avoids needing an entity extraction model.
    """
    return " ".join(question_item["keywords"])


def build_expected_answer_oracle_query(question_item):
    """
    Diagnostic only.
    This should NOT be used in the final game-playing chatbot because it uses the known answer.
    It helps estimate whether the retriever can find good evidence if guided correctly.
    """
    return f"{' '.join(question_item['keywords'])} {question_item['expected_answer']}"


def generate_query_variants(question_item):
    return {
        "raw_question": question_item["question"],
        "keyword_condensed": build_keyword_condensed_query(question_item),
        "domain_prefixed": build_domain_prefixed_query(question_item),
        "entity_focused": build_entity_focused_query(question_item),
        "expected_answer_oracle": build_expected_answer_oracle_query(question_item)
    }

### Preview query strategies

In [ ]:
query_preview_rows = []

for q in retrieval_test_questions:
    variants = generate_query_variants(q)

    for strategy_name, query_text in variants.items():
        query_preview_rows.append({
            "question_id": q["id"],
            "competition": q["competition"],
            "question_type": q["question_type"],
            "strategy": strategy_name,
            "query": query_text
        })

query_preview_df = pd.DataFrame(query_preview_rows)

query_preview_df.head(25)

,question_id,competition,question_type,strategy,query
0,AHP_Q1,Ancient History and Politics,direct_entity,raw_question,What term describes the region centered around...
1,AHP_Q1,Ancient History and Politics,direct_entity,keyword_condensed,describes region centered around city Babylon ...
2,AHP_Q1,Ancient History and Politics,direct_entity,domain_prefixed,ancient history politics What term describes t...
3,AHP_Q1,Ancient History and Politics,direct_entity,entity_focused,Babylon central-southern Mesopotamia Babylonia
4,AHP_Q1,Ancient History and Politics,direct_entity,expected_answer_oracle,Babylon central-southern Mesopotamia Babylonia...
5,AHP_Q2,Ancient History and Politics,artifact_material,raw_question,What is the primary material used to cast Roma...
6,AHP_Q2,Ancient History and Politics,artifact_material,keyword_condensed,material used cast Roman dodecahedra
7,AHP_Q2,Ancient History and Politics,artifact_material,domain_prefixed,ancient history politics What is the primary m...
8,AHP_Q2,Ancient History and Politics,artifact_material,entity_focused,Roman dodecahedra cast copper alloy bronze
9,AHP_Q2,Ancient History and Politics,artifact_material,expected_answer_oracle,Roman dodecahedra cast copper alloy bronze Cop...


In [ ]:
strategy_test_question_ids = [
    "AHP_Q1",   # direct factual history
    "AHP_Q4",   # conceptual history
    "ENT_Q4",   # entertainment/media reasoning
    "SCI_Q7",   # science formula
    "MATH_Q6"   # math/statistical reasoning
]

strategy_test_questions = [
    q for q in retrieval_test_questions
    if q["id"] in strategy_test_question_ids
]

len(strategy_test_questions)

5

In [ ]:
def run_query_strategy_benchmark(
    questions,
    retriever_name,
    retriever_function,
    strategies_to_test=None,
    max_results=3,
    sleep_seconds=1.0
):
    if strategies_to_test is None:
        strategies_to_test = [
            "raw_question",
            "keyword_condensed",
            "domain_prefixed",
            "entity_focused",
            "expected_answer_oracle"
        ]

    all_results = []

    for q in questions:
        query_variants = generate_query_variants(q)

        for strategy_name in strategies_to_test:
            query_text = query_variants[strategy_name]

            print(f"Processing {q['id']} | {retriever_name} | {strategy_name}")

            results = retriever_function(
                question_item=q,
                query_text=query_text,
                retriever_label=f"{retriever_name} - {strategy_name}",
                max_results=max_results
            )

            all_results.extend(results)

            time.sleep(sleep_seconds)

    return all_results

### Start with duckduckgo

In [ ]:
def retrieve_from_duckduckgo_query(
    question_item,
    query_text,
    retriever_label="DuckDuckGo",
    max_results=3
):
    question_id = question_item["id"]

    start_time = time.time()

    try:
        with DDGS() as ddgs:
            search_results = list(
                ddgs.text(
                    query_text,
                    max_results=max_results
                )
            )

    except Exception as e:
        print(f"DuckDuckGo failed for {question_id}: {e}")
        return []

    latency = time.time() - start_time

    results = []

    for rank, item in enumerate(search_results, start=1):

        title = item.get("title", "")
        url = item.get("href", "")
        content = item.get("body", "")

        results.append(
            RetrievalResult(
                retriever=retriever_label,
                question_id=question_id,
                query=query_text,
                title=title,
                url=url,
                content=content,
                rank=rank,
                latency_seconds=latency
            )
        )

    return results

In [ ]:
ddg_strategy_results = run_query_strategy_benchmark(
    questions=strategy_test_questions,
    retriever_name="DuckDuckGo",
    retriever_function=retrieve_from_duckduckgo_query,
    max_results=3,
    sleep_seconds=1.0
)

print(f"Total strategy benchmark results: {len(ddg_strategy_results)}")

Processing AHP_Q1 | DuckDuckGo | raw_question
Processing AHP_Q1 | DuckDuckGo | keyword_condensed
Processing AHP_Q1 | DuckDuckGo | domain_prefixed
Processing AHP_Q1 | DuckDuckGo | entity_focused
Processing AHP_Q1 | DuckDuckGo | expected_answer_oracle
Processing AHP_Q4 | DuckDuckGo | raw_question
Processing AHP_Q4 | DuckDuckGo | keyword_condensed
Processing AHP_Q4 | DuckDuckGo | domain_prefixed
Processing AHP_Q4 | DuckDuckGo | entity_focused
Processing AHP_Q4 | DuckDuckGo | expected_answer_oracle
Processing ENT_Q4 | DuckDuckGo | raw_question
Processing ENT_Q4 | DuckDuckGo | keyword_condensed
Processing ENT_Q4 | DuckDuckGo | domain_prefixed
Processing ENT_Q4 | DuckDuckGo | entity_focused
Processing ENT_Q4 | DuckDuckGo | expected_answer_oracle
Processing SCI_Q7 | DuckDuckGo | raw_question
Processing SCI_Q7 | DuckDuckGo | keyword_condensed
Processing SCI_Q7 | DuckDuckGo | domain_prefixed
Processing SCI_Q7 | DuckDuckGo | entity_focused
Processing SCI_Q7 | DuckDuckGo | expected_answer_oracle


In [ ]:
def retrieval_results_to_dataframe(results):

    rows = []

    for r in results:

        rows.append({
            "retriever": r.retriever,
            "question_id": r.question_id,
            "query": r.query,
            "title": r.title,
            "url": r.url,
            "content": r.content,
            "rank": r.rank,
            "latency_seconds": r.latency_seconds
        })

    return pd.DataFrame(rows)

In [ ]:
ddg_strategy_df = retrieval_results_to_dataframe(ddg_strategy_results)

print(ddg_strategy_df.shape)

ddg_strategy_df.head()

(75, 8)


,retriever,question_id,query,title,url,content,rank,latency_seconds
0,DuckDuckGo - raw_question,AHP_Q1,What term describes the region centered around...,Babylonia - Wikipedia,https://en.wikipedia.org/wiki/Babylonia,"However, in southern Mesopotamia (a region cor...",1,2.127138
1,DuckDuckGo - raw_question,AHP_Q1,What term describes the region centered around...,Assyrian Empire Builders - Babylon and the cit...,https://oracc.museum.upenn.edu/saao/aebp/Essen...,South of the Assyrian heartland lies Babylonia...,2,2.127138
2,DuckDuckGo - raw_question,AHP_Q1,What term describes the region centered around...,"Babylonia | History, Map, Culture, & Facts | B...",https://www.britannica.com/place/Babylonia,"Babylonia, ancient cultural region occupying s...",3,2.127138
3,DuckDuckGo - keyword_condensed,AHP_Q1,describes region centered around city Babylon ...,Babylonia - Wikipedia,https://en.wikipedia.org/wiki/Babylonia,2 weeks ago - It was often involved in rivalry...,1,2.551777
4,DuckDuckGo - keyword_condensed,AHP_Q1,describes region centered around city Babylon ...,Babylon - Wikipedia,https://en.wikipedia.org/wiki/Babylon,1 week ago - Babylon (/ˈbæbɪlɒn/ BAB-il-on) .....,2,2.551777


### Evaluate strategy results

In [ ]:
def evaluate_retrieval_results(retrieval_df, questions):
    """
    Evaluates the retrieval dataframe against expected answers from the questions list.
    Adds columns:
        - expected_answer
        - expected_answer_found (boolean)
        - keyword_hits
        - keyword_coverage
    """
    eval_rows = []
    # Build a lookup from question_id to expected answer
    expected_lookup = {q['id']: q.get('expected_answer', '') for q in questions}

    for _, row in retrieval_df.iterrows():
        qid = row['question_id']
        expected = expected_lookup.get(qid, "")
        content_lower = str(row['content']).lower()
        expected_lower = str(expected).lower()

        # Check if expected answer is present
        found = expected_lower in content_lower if expected else False

        # Keyword hits / coverage can be approximated as fraction of expected keywords present
        keywords = expected_lower.split()
        if keywords:
            hits = sum(1 for k in keywords if k in content_lower)
            coverage = hits / len(keywords)
        else:
            hits = 0
            coverage = 0.0

        eval_rows.append({
            **row.to_dict(),
            "expected_answer": expected,
            "expected_answer_found": found,
            "keyword_hits": hits,
            "keyword_coverage": coverage
        })

    return pd.DataFrame(eval_rows)

In [ ]:
ddg_strategy_eval_df = evaluate_retrieval_results(
    retrieval_df=ddg_strategy_df,
    questions=retrieval_test_questions
)

print(ddg_strategy_eval_df.shape)

ddg_strategy_eval_df.head(20)

(75, 12)


,retriever,question_id,query,title,url,content,rank,latency_seconds,expected_answer,expected_answer_found,keyword_hits,keyword_coverage
0,DuckDuckGo - raw_question,AHP_Q1,What term describes the region centered around...,Babylonia - Wikipedia,https://en.wikipedia.org/wiki/Babylonia,"However, in southern Mesopotamia (a region cor...",1,2.127138,Babylonia,False,0,0.0
1,DuckDuckGo - raw_question,AHP_Q1,What term describes the region centered around...,Assyrian Empire Builders - Babylon and the cit...,https://oracc.museum.upenn.edu/saao/aebp/Essen...,South of the Assyrian heartland lies Babylonia...,2,2.127138,Babylonia,True,1,1.0
2,DuckDuckGo - raw_question,AHP_Q1,What term describes the region centered around...,"Babylonia | History, Map, Culture, & Facts | B...",https://www.britannica.com/place/Babylonia,"Babylonia, ancient cultural region occupying s...",3,2.127138,Babylonia,True,1,1.0
3,DuckDuckGo - keyword_condensed,AHP_Q1,describes region centered around city Babylon ...,Babylonia - Wikipedia,https://en.wikipedia.org/wiki/Babylonia,2 weeks ago - It was often involved in rivalry...,1,2.551777,Babylonia,True,1,1.0
4,DuckDuckGo - keyword_condensed,AHP_Q1,describes region centered around city Babylon ...,Babylon - Wikipedia,https://en.wikipedia.org/wiki/Babylon,1 week ago - Babylon (/ˈbæbɪlɒn/ BAB-il-on) .....,2,2.551777,Babylonia,True,1,1.0
5,DuckDuckGo - keyword_condensed,AHP_Q1,describes region centered around city Babylon ...,Geography of Mesopotamia - Wikipedia,https://en.wikipedia.org/wiki/Geography_of_Mes...,"November 19, 2025 - The geography of Mesopotam...",3,2.551777,Babylonia,False,0,0.0
6,DuckDuckGo - domain_prefixed,AHP_Q1,ancient history politics What term describes t...,Babylonia - Wikipedia,https://en.wikipedia.org/wiki/Babylonia,"Babylonia (/ ˌbæbɪˈloʊniə /; Akkadian: 𒆳𒆍𒀭𒊏𒆠, ...",1,1.704969,Babylonia,True,1,1.0
7,DuckDuckGo - domain_prefixed,AHP_Q1,ancient history politics What term describes t...,Babylon and the cities and tribes of Southern ...,https://oracc.museum.upenn.edu/saao/aebp/essen...,"Apr 23, 2024 · In 729 BC, Tiglatpileser III (7...",2,1.704969,Babylonia,True,1,1.0
8,DuckDuckGo - domain_prefixed,AHP_Q1,ancient history politics What term describes t...,"Babylon | History, Religion, Time Period, & Fa...",https://www.britannica.com/place/Babylon-ancie...,"Hammurabi (1792–1750 BCE), the sixth and best-...",3,1.704969,Babylonia,True,1,1.0
9,DuckDuckGo - entity_focused,AHP_Q1,Babylon central-southern Mesopotamia Babylonia,Babylonia - Wikipedia,https://en.wikipedia.org/wiki/Babylonia,"Babylonia (/ ˌbæbɪˈloʊniə /; Akkadian: 𒆳𒆍𒀭𒊏𒆠, ...",1,0.949589,Babylonia,True,1,1.0


### Strategy summary comparison

In [ ]:
ddg_strategy_summary_df = (
    ddg_strategy_eval_df
    .groupby(["question_id", "retriever"], dropna=False)
    .agg(
        total_results=("title", "count"),
        top_result_title=("title", "first"),
        top_result_expected_answer_found=("expected_answer_found", "first"),
        any_result_expected_answer_found=("expected_answer_found", "max"),
        max_keyword_coverage=("keyword_coverage", "max"),
        avg_latency_seconds=("latency_seconds", "mean")
    )
    .reset_index()
)

ddg_strategy_summary_df

,question_id,retriever,total_results,top_result_title,top_result_expected_answer_found,any_result_expected_answer_found,max_keyword_coverage,avg_latency_seconds
0,AHP_Q1,DuckDuckGo - domain_prefixed,3,Babylonia - Wikipedia,True,True,1.000000,1.704969
1,AHP_Q1,DuckDuckGo - entity_focused,3,Babylonia - Wikipedia,True,True,1.000000,0.949589
2,AHP_Q1,DuckDuckGo - expected_answer_oracle,3,Babylonia - Wikipedia,True,True,1.000000,1.270250
3,AHP_Q1,DuckDuckGo - keyword_condensed,3,Babylonia - Wikipedia,True,True,1.000000,2.551777
4,AHP_Q1,DuckDuckGo - raw_question,3,Babylonia - Wikipedia,False,True,1.000000,2.127138
5,AHP_Q4,DuckDuckGo - domain_prefixed,3,Caesarism - Wikipedia,False,False,0.400000,1.299130
6,AHP_Q4,DuckDuckGo - entity_focused,3,Caesarism - Wikipedia,False,False,0.900000,1.583500
7,AHP_Q4,DuckDuckGo - expected_answer_oracle,3,Caesarism - Wikipedia,False,False,0.800000,1.174993
8,AHP_Q4,DuckDuckGo - keyword_condensed,3,Caesarism - Wikipedia,False,False,0.400000,1.272650
9,AHP_Q4,DuckDuckGo - raw_question,3,Caesarism - Wikipedia,False,False,0.500000,0.833534


In [ ]:
strategy_summary_df = (
    ddg_strategy_summary_df
    .groupby("retriever")
    .agg(
        questions_tested=("question_id", "nunique"),
        questions_hit=("any_result_expected_answer_found", "sum"),
        avg_keyword_coverage=("max_keyword_coverage", "mean"),
        avg_latency=("avg_latency_seconds", "mean")
    )
    .reset_index()
)

strategy_summary_df["answer_found_rate"] = (
    strategy_summary_df["questions_hit"]
    / strategy_summary_df["questions_tested"]
).round(3)

strategy_summary_df = strategy_summary_df.sort_values(
    by=["answer_found_rate", "avg_keyword_coverage"],
    ascending=[False, False]
)

strategy_summary_df

,retriever,questions_tested,questions_hit,avg_keyword_coverage,avg_latency,answer_found_rate
2,DuckDuckGo - expected_answer_oracle,5,2,0.786738,1.422418,0.4
0,DuckDuckGo - domain_prefixed,5,2,0.765561,1.874205,0.4
4,DuckDuckGo - raw_question,5,2,0.726738,1.786209,0.4
1,DuckDuckGo - entity_focused,5,1,0.624920,1.441352,0.2
3,DuckDuckGo - keyword_condensed,5,1,0.435080,1.682102,0.2


### Question-level comparison

In [ ]:
strategy_question_comparison_df = (
    ddg_strategy_summary_df[
        [
            "question_id",
            "retriever",
            "top_result_title",
            "any_result_expected_answer_found",
            "max_keyword_coverage"
        ]
    ]
    .sort_values(
        by=["question_id", "max_keyword_coverage"],
        ascending=[True, False]
    )
)

strategy_question_comparison_df

,question_id,retriever,top_result_title,any_result_expected_answer_found,max_keyword_coverage
0,AHP_Q1,DuckDuckGo - domain_prefixed,Babylonia - Wikipedia,True,1.000000
1,AHP_Q1,DuckDuckGo - entity_focused,Babylonia - Wikipedia,True,1.000000
2,AHP_Q1,DuckDuckGo - expected_answer_oracle,Babylonia - Wikipedia,True,1.000000
3,AHP_Q1,DuckDuckGo - keyword_condensed,Babylonia - Wikipedia,True,1.000000
4,AHP_Q1,DuckDuckGo - raw_question,Babylonia - Wikipedia,True,1.000000
6,AHP_Q4,DuckDuckGo - entity_focused,Caesarism - Wikipedia,False,0.900000
7,AHP_Q4,DuckDuckGo - expected_answer_oracle,Caesarism - Wikipedia,False,0.800000
9,AHP_Q4,DuckDuckGo - raw_question,Caesarism - Wikipedia,False,0.500000
5,AHP_Q4,DuckDuckGo - domain_prefixed,Caesarism - Wikipedia,False,0.400000
8,AHP_Q4,DuckDuckGo - keyword_condensed,Caesarism - Wikipedia,False,0.400000


### Query Strategy Benchmarking — Findings & Conclusion

In this section, we tested whether retrieval quality improves when the same question is reformulated using different query strategies.

The strategies tested were:

- `raw_question`
- `keyword_condensed`
- `domain_prefixed`
- `entity_focused`
- `expected_answer_oracle`

The experiment was run first on DuckDuckGo to avoid wasting limited Tavily or SerpAPI credits.

### Main Findings

Query formulation has a clear impact on retrieval behavior.

The strongest practical strategies were:

- `domain_prefixed`
- `entity_focused`
- `raw_question`

The `expected_answer_oracle` strategy worked as an upper-bound diagnostic, but it cannot be used in the real chatbot because it uses the known correct answer.

The `keyword_condensed` strategy was less stable than expected. In some cases, it removed too much context and produced weaker retrieval.

### Key Insight

This experiment confirmed that retrieval quality is not only determined by the search engine. It is also strongly affected by how the query is written.

However, the experiment also showed that even improved queries do not fully solve reasoning-heavy questions.

Some questions require:

- semantic interpretation
- comparison reasoning
- statistical reasoning
- mathematical computation
- or tool-based solving

### Conclusion

The retrieval layer is useful, but it cannot be evaluated only with keyword overlap or exact answer matching.

The next section introduces a local LLM-based judge to evaluate whether retrieved evidence is actually useful for answering the question.

## LLM-Based Retrieval Judging



In the previous sections, we evaluated retrievers using lexical metrics such as:

- exact expected-answer matching
- keyword hits
- keyword coverage
- latency

These metrics are useful as a first benchmark, but they do not fully capture whether the retrieved context is actually helpful for answering the question.

In this section, we introduce a local open-weight LLM as a retrieval judge.

The goal is not yet to make the final chatbot answer the game questions.

Instead, the goal is to evaluate retrieved evidence more intelligently.

For each question and retrieved result, the LLM judge will assess:

- whether the retrieved context is relevant,
- whether it supports the correct answer,
- whether it is sufficient for answering,
- whether it is misleading,
- and how confident the judge is.

This will allow us to compare retrievers more fairly than simple keyword matching.

The next step is to switch to GPU runtime and load the local Mistral model used earlier in the notebook.

In [ ]:
# Load Mistral model
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Format messages (Mistral uses chat template too)
messages = [
    {"role": "user", "content": "Who are you?"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# Generate
outputs = model.generate(
    **inputs,
    max_new_tokens=40,
    do_sample=False
)

# Decode only generated part
response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


I am a large language model trained by Mistral AI. I am designed to generate human-like text based on the input I receive. I don't have the ability to have feelings, emotions


In [ ]:
def generate_mistral_response(
    prompt,
    max_new_tokens=256,
    temperature=0.0
):
    """
    Simple reusable inference wrapper for Mistral.
    """

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=temperature > 0,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
test_prompt = """
What is the capital of France?
Answer in one short sentence.
"""

response = generate_mistral_response(test_prompt)

print(response)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


The capital city of France is Paris.


### LLM judge prompt builder

In [ ]:
def build_retrieval_judge_prompt(question_item, retrieved_row):
    """
    Builds a prompt for judging whether one retrieved result is useful evidence
    for answering a multiple-choice question.
    """

    options_text = "\n".join([
        f"{i}. {option}"
        for i, option in enumerate(question_item["options"])
    ])

    prompt = f"""
You are evaluating whether a retrieved web result is useful for answering a multiple-choice quiz question.

Question:
{question_item["question"]}

Options:
{options_text}

Expected correct answer:
{question_item["expected_answer"]}

Retrieved result title:
{retrieved_row["title"]}

Retrieved result content:
{retrieved_row["content"]}

Evaluate the retrieved result.

Return ONLY valid JSON with these fields:
{{
  "is_relevant": true/false,
  "supports_correct_answer": true/false,
  "is_misleading": true/false,
  "evidence_quality": "none" | "weak" | "partial" | "strong",
  "reasoning_needed": "none" | "simple_lookup" | "comparison" | "multi_hop" | "math_or_tool",
  "brief_reason": "short explanation"
}}
"""
    return prompt

### Test on one retrieved result

In [ ]:
test_question_item = {
    "id": "AHP_Q1",
    "question": "What term describes the region centered around the city of Babylon in central-southern Mesopotamia?",
    "options": [
        "Assyria",
        "Babylonia",
        "Sumer",
        "Akkadia"
    ],
    "expected_answer": "Babylonia"
}

test_retrieved_row = {
    "title": "Babylonia - Wikipedia",
    "content": """
Babylonia was an ancient Akkadian-speaking state and cultural area
based on the city of Babylon in central-southern Mesopotamia.
"""
}

In [ ]:
judge_prompt = build_retrieval_judge_prompt(
    question_item=test_question_item,
    retrieved_row=test_retrieved_row
)

judge_response = generate_mistral_response(
    judge_prompt,
    max_new_tokens=250,
    temperature=0.0
)

print(judge_response)

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


{
  "is_relevant": true,
  "supports_correct_answer": true,
  "is_misleading": false,
  "evidence_quality": "strong",
  "reasoning_needed": "simple_lookup",
  "brief_reason": "The retrieved result is a Wikipedia page about Babylonia, which is the correct answer to the question. The content of the page clearly states that Babylonia is a region centered around the city of Babylon in central-southern Mesopotamia."
}


In [ ]:
import json

def parse_llm_json_response(response_text):
    """
    Safely parses LLM JSON responses.
    """

    try:
        parsed = json.loads(response_text)
        return parsed

    except Exception as e:

        print("JSON parsing failed")
        print(e)

        return {
            "is_relevant": None,
            "supports_correct_answer": None,
            "is_misleading": None,
            "evidence_quality": None,
            "reasoning_needed": None,
            "brief_reason": response_text
        }

In [ ]:
parsed_response = parse_llm_json_response(judge_response)

parsed_response

{'is_relevant': True,
 'supports_correct_answer': True,
 'is_misleading': False,
 'evidence_quality': 'strong',
 'reasoning_needed': 'simple_lookup',
 'brief_reason': 'The retrieved result is a Wikipedia page about Babylonia, which is the correct answer to the question. The content of the page clearly states that Babylonia is a region centered around the city of Babylon in central-southern Mesopotamia.'}

### helper to get question item

In [ ]:
def get_question_item(question_id):
    return next(
        q for q in retrieval_test_questions
        if q["id"] == question_id
    )

### reusable judge function

In [ ]:
def judge_retrieval_result_with_llm(row):
    """
    Uses Mistral to judge one retrieved result row.
    """

    question_item = get_question_item(row["question_id"])

    judge_prompt = build_retrieval_judge_prompt(
        question_item=question_item,
        retrieved_row=row
    )

    response = generate_mistral_response(
        judge_prompt,
        max_new_tokens=250,
        temperature=0.0
    )

    parsed = parse_llm_json_response(response)

    return parsed

### Fetching data

In [ ]:
import pandas as pd

def retrieval_results_to_dataframe(results):
    """
    Converts RetrievalResult objects into a pandas DataFrame.
    """

    rows = []

    for r in results:
        rows.append({
            "retriever": r.retriever,
            "question_id": r.question_id,
            "query": r.query,
            "title": r.title,
            "url": r.url,
            "content": r.content,
            "rank": r.rank,
            "latency_seconds": r.latency_seconds
        })

    return pd.DataFrame(rows)

In [ ]:
wiki_safe_df = retrieval_results_to_dataframe(wiki_safe_results)
ddg_df = retrieval_results_to_dataframe(ddg_results)

In [ ]:
wiki_safe_df.columns
ddg_df.columns

Index(['retriever', 'question_id', 'query', 'title', 'url', 'content', 'rank',
       'latency_seconds'],
      dtype='object')

### create small sample from Wikipedia + DuckDuckGo

In [ ]:
llm_judge_sample_df = pd.concat(
    [
        wiki_safe_df.head(5),
        ddg_df.head(5)
    ],
    ignore_index=True
)

llm_judge_sample_df[
    [
        "retriever",
        "question_id",
        "rank",
        "title"
    ]
]

,retriever,question_id,rank,title
0,Wikipedia Safe,AHP_Q1,1,Mesopotamia
1,Wikipedia Safe,AHP_Q1,2,History of Mesopotamia
2,Wikipedia Safe,AHP_Q1,3,Babylonia
3,Wikipedia Safe,AHP_Q3,1,Homeric Hymns
4,Wikipedia Safe,AHP_Q3,2,Homer
5,DuckDuckGo,AHP_Q1,1,Babylonia - Wikipedia
6,DuckDuckGo,AHP_Q1,2,Mesopotamia - Wikipedia
7,DuckDuckGo,AHP_Q1,3,Babylon - Wikipedia
8,DuckDuckGo,AHP_Q1,4,Geography of Mesopotamia - Wikipedia
9,DuckDuckGo,AHP_Q1,5,Sumer - Wikipedia


### run LLM judge on the sample

In [ ]:
llm_judge_rows = []

for idx, row in llm_judge_sample_df.iterrows():

    print(
        f"Judging row {idx+1}/{len(llm_judge_sample_df)} | "
        f"{row['retriever']} | {row['question_id']}"
    )

    judgement = judge_retrieval_result_with_llm(row)

    llm_judge_rows.append({
        **row.to_dict(),
        **judgement
    })

llm_judge_sample_results_df = pd.DataFrame(llm_judge_rows)

llm_judge_sample_results_df[
    [
        "retriever",
        "question_id",
        "rank",
        "title",
        "is_relevant",
        "supports_correct_answer",
        "is_misleading",
        "evidence_quality",
        "reasoning_needed",
        "brief_reason"
    ]
]

Judging row 1/10 | Wikipedia Safe | AHP_Q1
Judging row 2/10 | Wikipedia Safe | AHP_Q1
Judging row 3/10 | Wikipedia Safe | AHP_Q1
Judging row 4/10 | Wikipedia Safe | AHP_Q3
Judging row 5/10 | Wikipedia Safe | AHP_Q3
Judging row 6/10 | DuckDuckGo | AHP_Q1
Judging row 7/10 | DuckDuckGo | AHP_Q1
Judging row 8/10 | DuckDuckGo | AHP_Q1
Judging row 9/10 | DuckDuckGo | AHP_Q1
Judging row 10/10 | DuckDuckGo | AHP_Q1


,retriever,question_id,rank,title,is_relevant,supports_correct_answer,is_misleading,evidence_quality,reasoning_needed,brief_reason
0,Wikipedia Safe,AHP_Q1,1,Mesopotamia,True,True,False,strong,simple_lookup,The retrieved result mentions 'Babylonia' as o...
1,Wikipedia Safe,AHP_Q1,2,History of Mesopotamia,True,True,False,strong,none,"The retrieved result, 'History of Mesopotamia'..."
2,Wikipedia Safe,AHP_Q1,3,Babylonia,True,True,False,strong,none,The retrieved result directly states that Baby...
3,Wikipedia Safe,AHP_Q3,1,Homeric Hymns,True,False,False,none,none,The retrieved result discusses the Homeric Hym...
4,Wikipedia Safe,AHP_Q3,2,Homer,True,False,False,none,none,"The retrieved result discusses Homer, an ancie..."
5,DuckDuckGo,AHP_Q1,1,Babylonia - Wikipedia,True,True,False,strong,none,The retrieved result is a Wikipedia page about...
6,DuckDuckGo,AHP_Q1,2,Mesopotamia - Wikipedia,True,True,False,strong,simple_lookup,The retrieved result clearly states that Babyl...
7,DuckDuckGo,AHP_Q1,3,Babylon - Wikipedia,True,True,False,strong,simple_lookup,"The retrieved result, Babylon - Wikipedia, dir..."
8,DuckDuckGo,AHP_Q1,4,Geography of Mesopotamia - Wikipedia,True,True,False,strong,simple_lookup,The retrieved result from Wikipedia provides a...
9,DuckDuckGo,AHP_Q1,5,Sumer - Wikipedia,True,False,False,weak,simple_lookup,The retrieved result mentions Babylonia in rel...


### More questions judging

In [ ]:
# Build full LLM judge dataset from Wikipedia + DuckDuckGo
# Keep only top 3 results per question per retriever

llm_judge_full_df = pd.concat(
    [
        wiki_safe_df,
        ddg_df
    ],
    ignore_index=True
)

llm_judge_full_df = (
    llm_judge_full_df
    .sort_values(["retriever", "question_id", "rank"])
    .groupby(["retriever", "question_id"], as_index=False)
    .head(3)
    .reset_index(drop=True)
)

print(llm_judge_full_df.shape)

llm_judge_full_df[
    ["retriever", "question_id", "rank", "title"]
].head(20)

(138, 8)


,retriever,question_id,rank,title
0,DuckDuckGo,AHP_Q1,1,Babylonia - Wikipedia
1,DuckDuckGo,AHP_Q1,2,Mesopotamia - Wikipedia
2,DuckDuckGo,AHP_Q1,3,Babylon - Wikipedia
3,DuckDuckGo,AHP_Q2,1,Roman dodecahedron - Wikipedia
4,DuckDuckGo,AHP_Q2,2,The Mystery of the Ancient Roman Dodecahedrons...
5,DuckDuckGo,AHP_Q2,3,The Dodecahedron: Ancient Toy or Practical Too...
6,DuckDuckGo,AHP_Q3,1,Homer - Wikipedia
7,DuckDuckGo,AHP_Q3,2,Introduction – The Battle of Frogs and Mice
8,DuckDuckGo,AHP_Q3,3,(PDF) A Chronological Study of the Editions of...
9,DuckDuckGo,AHP_Q4,1,Caesarism - Wikipedia


In [ ]:
llm_judge_rows = []

for idx, row in llm_judge_full_df.iterrows():

    print(
        f"Judging row {idx+1}/{len(llm_judge_full_df)} | "
        f"{row['retriever']} | {row['question_id']} | rank {row['rank']}"
    )

    judgement = judge_retrieval_result_with_llm(row)

    llm_judge_rows.append({
        **row.to_dict(),
        **judgement
    })

llm_judge_results_df = pd.DataFrame(llm_judge_rows)

llm_judge_results_df.head()

Judging row 1/138 | DuckDuckGo | AHP_Q1 | rank 1
Judging row 2/138 | DuckDuckGo | AHP_Q1 | rank 2
Judging row 3/138 | DuckDuckGo | AHP_Q1 | rank 3
Judging row 4/138 | DuckDuckGo | AHP_Q2 | rank 1
Judging row 5/138 | DuckDuckGo | AHP_Q2 | rank 2
Judging row 6/138 | DuckDuckGo | AHP_Q2 | rank 3
Judging row 7/138 | DuckDuckGo | AHP_Q3 | rank 1
Judging row 8/138 | DuckDuckGo | AHP_Q3 | rank 2
Judging row 9/138 | DuckDuckGo | AHP_Q3 | rank 3
Judging row 10/138 | DuckDuckGo | AHP_Q4 | rank 1
Judging row 11/138 | DuckDuckGo | AHP_Q4 | rank 2
Judging row 12/138 | DuckDuckGo | AHP_Q4 | rank 3
Judging row 13/138 | DuckDuckGo | AHP_Q5 | rank 1
Judging row 14/138 | DuckDuckGo | AHP_Q5 | rank 2
Judging row 15/138 | DuckDuckGo | AHP_Q5 | rank 3
Judging row 16/138 | DuckDuckGo | AHP_Q6 | rank 1
Judging row 17/138 | DuckDuckGo | AHP_Q6 | rank 2
Judging row 18/138 | DuckDuckGo | AHP_Q6 | rank 3
Judging row 19/138 | DuckDuckGo | ENT_Q1 | rank 1
Judging row 20/138 | DuckDuckGo | ENT_Q1 | rank 2
Judging r

,retriever,question_id,query,title,url,content,rank,latency_seconds,is_relevant,supports_correct_answer,is_misleading,evidence_quality,reasoning_needed,brief_reason
0,DuckDuckGo,AHP_Q1,What term describes the region centered around...,Babylonia - Wikipedia,https://en.wikipedia.org/wiki/Babylonia,2 weeks ago - Babylonia (/ˌbæbɪˈloʊniə/; Akkad...,1,1.575645,True,True,False,strong,none,The retrieved result is a Wikipedia page about...
1,DuckDuckGo,AHP_Q1,What term describes the region centered around...,Mesopotamia - Wikipedia,https://en.wikipedia.org/wiki/Mesopotamia,5 days ago - Although Babylon was quite a smal...,2,1.575645,True,True,False,strong,simple_lookup,The retrieved result clearly states that Babyl...
2,DuckDuckGo,AHP_Q1,What term describes the region centered around...,Babylon - Wikipedia,https://en.wikipedia.org/wiki/Babylon,1 week ago - Babylon (/ˈbæbɪlɒn/ BAB-il-on) .....,3,1.575645,True,True,False,strong,simple_lookup,"The retrieved result, Babylon - Wikipedia, dir..."
3,DuckDuckGo,AHP_Q2,What is the primary material used to cast Roma...,Roman dodecahedron - Wikipedia,https://en.wikipedia.org/wiki/Roman_dodecahedron,3 weeks ago - A Roman dodecahedron or Gallo-Ro...,1,1.661401,True,True,False,strong,simple_lookup,The retrieved result directly states that Roma...
4,DuckDuckGo,AHP_Q2,What is the primary material used to cast Roma...,The Mystery of the Ancient Roman Dodecahedrons...,https://thequantumrecord.com/technology-over-t...,"November 28, 2024 - The rings probably contain...",2,1.661401,True,False,False,none,none,The text mentions the use of natural fibers fo...


In [ ]:
llm_judge_summary_df = (
    llm_judge_results_df
    .groupby(["retriever", "question_id"])
    .agg(
        judged_results=("title", "count"),
        relevant_results=("is_relevant", "sum"),
        supporting_results=("supports_correct_answer", "sum"),
        misleading_results=("is_misleading", "sum"),
        best_evidence_quality=("evidence_quality", lambda x: list(x)),
        top_result_title=("title", "first"),
        top_result_supports_answer=("supports_correct_answer", "first")
    )
    .reset_index()
)

llm_judge_summary_df

,retriever,question_id,judged_results,relevant_results,supporting_results,misleading_results,best_evidence_quality,top_result_title,top_result_supports_answer
0,DuckDuckGo,AHP_Q1,3,3,3,0,"[strong, strong, strong]",Babylonia - Wikipedia,True
1,DuckDuckGo,AHP_Q2,3,3,2,0,"[strong, none, strong]",Roman dodecahedron - Wikipedia,True
2,DuckDuckGo,AHP_Q3,3,3,2,0,"[strong, weak, strong]",Homer - Wikipedia,True
3,DuckDuckGo,AHP_Q4,3,3,3,0,"[strong, strong, strong]",Caesarism - Wikipedia,True
4,DuckDuckGo,AHP_Q5,3,3,3,0,"[strong, strong, strong]",Classical order - Wikipedia,True
5,DuckDuckGo,AHP_Q6,3,3,3,0,"[strong, strong, strong]","(PDF) The Antikythera Mechanism, Rhodes, and E...",True
6,DuckDuckGo,ENT_Q1,3,3,3,0,"[strong, strong, strong]",Scat singing - Wikipedia,True
7,DuckDuckGo,ENT_Q2,3,3,3,0,"[strong, strong, strong]",Voice type - Wikipedia,True
8,DuckDuckGo,ENT_Q3,3,3,3,0,"[strong, strong, strong]",Death of Marilyn Monroe - Wikipedia,True
9,DuckDuckGo,ENT_Q4,3,3,2,0,"[none, strong, strong]",MTV - Wikipedia,False


In [ ]:
llm_retriever_comparison_df = (
    llm_judge_summary_df
    .groupby("retriever")
    .agg(
        questions_judged=("question_id", "nunique"),
        questions_with_supporting_evidence=("supporting_results", lambda x: (x > 0).sum()),
        avg_supporting_results=("supporting_results", "mean"),
        avg_relevant_results=("relevant_results", "mean"),
        avg_misleading_results=("misleading_results", "mean")
    )
    .reset_index()
)

llm_retriever_comparison_df["llm_support_rate"] = (
    llm_retriever_comparison_df["questions_with_supporting_evidence"]
    / llm_retriever_comparison_df["questions_judged"]
).round(3)

llm_retriever_comparison_df

,retriever,questions_judged,questions_with_supporting_evidence,avg_supporting_results,avg_relevant_results,avg_misleading_results,llm_support_rate
0,DuckDuckGo,25,19,1.840000,2.76000,0.040000,0.760
1,Wikipedia Safe,21,13,1.095238,2.47619,0.142857,0.619


### LLM-Based Retrieval Judging — Findings & Conclusion

In this section, we introduced a local open-weight Mistral model as a retrieval judge.

The goal was not to answer the quiz questions yet, but to evaluate whether retrieved evidence was actually useful for answering them.

This moved the evaluation beyond simple lexical metrics such as:

- exact expected-answer matching
- keyword hits
- keyword coverage

### Why This Was Needed

The previous retrieval benchmarks showed that exact matching often underestimates retrieval quality.

A retrieved page may be useful even if it does not contain the exact expected answer phrase.

Similarly, a retrieved page may contain a keyword or answer string but still be irrelevant or misleading.

### LLM Judge Results

Using the LLM judge, retrieval quality appeared much stronger than the lexical evaluator suggested.

| Retriever | Questions Judged | Questions With Supporting Evidence | LLM Support Rate |
|---|---:|---:|---:|
| DuckDuckGo | 25 | 19 | 76.0% |
| Wikipedia Safe | 21 | 13 | 61.9% |

### Main Findings

DuckDuckGo showed stronger retrieval usefulness than Wikipedia Safe when judged semantically by the LLM.

DuckDuckGo achieved:

- higher support rate
- more supporting results per question
- lower misleading-result rate
- broader question coverage

Wikipedia Safe remained useful for factual and encyclopedic questions, but struggled more when the needed evidence required broader web context.

### Important Insight

This experiment confirmed that:

> Retrieval quality is better measured by evidence usefulness than by exact answer string matching.

The LLM judge helped separate:

1. retrieval failure,
2. evaluation failure,
3. reasoning/tool-use failure.

This is especially important for Maths questions, where retrieved pages may be topically relevant but still insufficient for solving the problem.

### Conclusion

The LLM judge provides a stronger and more realistic way to evaluate retrieval quality.

The next step is to move from judging retrieval evidence to actually using the retrieved evidence for answering the multiple-choice questions.

## Retrieval-Augmented Question Answering with Local LLM



In the previous section, the local Mistral model acted as a judge.

It evaluated whether retrieved evidence was useful, but it did not select final answers.

In this section, the LLM becomes the actual solver.

For each question, we provide:

- the question text,
- the multiple-choice options,
- retrieved evidence from Wikipedia Safe or DuckDuckGo,
- and ask the model to select the correct option.

The model must answer using only the retrieved context and the provided options.

The goal is to evaluate whether retrieval actually improves multiple-choice answering performance.

We will begin with a small sample and compare:

1. Wikipedia-based retrieval-augmented answering,
2. DuckDuckGo-based retrieval-augmented answering,
3. later possibly Tavily or SerpAPI on selected questions only.

The evaluation metric will be simple:

- predicted option,
- expected option,
- correct / incorrect.

In [ ]:
DOMAIN_SOLVER_CONTEXT = {
    "Ancient History and Politics": "The question is from ancient history, archaeology, classical studies, or political concepts.",
    "Entertainment": "The question is from entertainment, music, film, celebrities, or media history.",
    "Science and Nature": "The question is from science, nature, biology, chemistry, physics, astronomy, or environmental reasoning.",
    "Maths": "The question is from mathematics, statistics, geometry, logic, or quantitative reasoning."
}


def build_retrieval_augmented_solver_prompt(question_item, retrieved_rows):
    options_text = "\n".join([
        f"{i}. {option}"
        for i, option in enumerate(question_item["options"])
    ])

    domain_context = DOMAIN_SOLVER_CONTEXT.get(
        question_item["competition"],
        "The question belongs to a general knowledge quiz category."
    )

    evidence_blocks = []

    for idx, row in enumerate(retrieved_rows, start=1):
        evidence_blocks.append(
            f"""
Evidence {idx}
Title: {row["title"]}
Source: {row.get("url", "")}
Content:
{row["content"]}
"""
        )

    evidence_text = "\n".join(evidence_blocks)

    prompt = f"""
You are answering a multiple-choice quiz question.

Competition/domain:
{question_item["competition"]}

Domain guidance:
{domain_context}

Question:
{question_item["question"]}

Options:
{options_text}

Retrieved evidence:
{evidence_text}

Instructions:
- Use the retrieved evidence when it is relevant.
- If the evidence is weak, use careful reasoning based on the question and options.
- Choose the single best option.
- Return ONLY valid JSON.
- Do not include markdown.

JSON format:
{{
  "predicted_option": 0,
  "predicted_answer": "answer text",
  "confidence": "low | medium | high",
  "used_evidence": true,
  "brief_reason": "short explanation"
}}
"""
    return prompt

In [ ]:
test_question_id = "AHP_Q1"

question_item = next(
    q for q in retrieval_test_questions
    if q["id"] == test_question_id
)

retrieved_rows = (
    ddg_df[ddg_df["question_id"] == test_question_id]
    .sort_values("rank")
    .head(3)
    .to_dict("records")
)

solver_prompt = build_retrieval_augmented_solver_prompt(
    question_item=question_item,
    retrieved_rows=retrieved_rows
)

solver_response = generate_mistral_response(
    solver_prompt,
    max_new_tokens=250,
    temperature=0.0
)

print(solver_response)

{
  "predicted_option": 2,
  "predicted_answer": "Babylonia",
  "confidence": "high",
  "used_evidence": true,
  "brief_reason": "All three pieces of evidence clearly indicate that Babylonia is the correct answer. The term 'Babylonia' refers to the region centered around the city of Babylon in Mesopotamia."
}


### Answering more questions using duckduckgo

In [ ]:
def build_rag_solver_number_only_prompt(question_item, retrieved_rows):
    options_text = "\n".join([
        f"{i}. {option}"
        for i, option in enumerate(question_item["options"])
    ])

    domain_context = DOMAIN_SOLVER_CONTEXT.get(
        question_item["competition"],
        "The question belongs to a general knowledge quiz category."
    )

    evidence_blocks = []

    for idx, row in enumerate(retrieved_rows, start=1):
        evidence_blocks.append(
            f"""
Evidence {idx}
Title: {row["title"]}
Content:
{row["content"]}
"""
        )

    evidence_text = "\n".join(evidence_blocks)

    prompt = f"""
You are answering a multiple-choice quiz question.

Competition/domain:
{question_item["competition"]}

Domain guidance:
{domain_context}

Question:
{question_item["question"]}

Options:
{options_text}

Retrieved evidence:
{evidence_text}

Rules:
- Choose the single best option.
- Answer using ONLY the option number.
- The option number must be one of: 0, 1, 2, or 3.
- Do NOT explain.
- Do NOT write any words.
- Output a single digit only.

Answer:
"""
    return prompt

In [ ]:
import re
import pandas as pd
import time

ddg_rag_solver_rows = []

for idx, question_item in enumerate(retrieval_test_questions):

    question_id = question_item["id"]

    print(f"Solving {idx+1}/{len(retrieval_test_questions)} | {question_id}")

    retrieved_rows = (
        ddg_df[ddg_df["question_id"] == question_id]
        .sort_values("rank")
        .head(3)
        .to_dict("records")
    )

    if len(retrieved_rows) == 0:
        selected_option = None
        raw_response = ""
    else:
        solver_prompt = build_rag_solver_number_only_prompt(
            question_item=question_item,
            retrieved_rows=retrieved_rows
        )

        raw_response = generate_mistral_response(
            solver_prompt,
            max_new_tokens=5,
            temperature=0.0
        )

        match = re.search(r"[0-3]", raw_response)
        selected_option = int(match.group()) if match else None

    ddg_rag_solver_rows.append({
        "retriever": "DuckDuckGo",
        "question_id": question_id,
        "competition": question_item["competition"],
        "question_type": question_item["question_type"],
        "selected_option": selected_option,
        "raw_response": raw_response
    })

    time.sleep(0.2)

ddg_rag_solver_df = pd.DataFrame(ddg_rag_solver_rows)

ddg_rag_solver_df

Solving 1/25 | AHP_Q1
Solving 2/25 | AHP_Q2
Solving 3/25 | AHP_Q3
Solving 4/25 | AHP_Q4
Solving 5/25 | AHP_Q5
Solving 6/25 | AHP_Q6
Solving 7/25 | ENT_Q1
Solving 8/25 | ENT_Q2
Solving 9/25 | ENT_Q3
Solving 10/25 | ENT_Q4
Solving 11/25 | ENT_Q5
Solving 12/25 | SCI_Q1
Solving 13/25 | SCI_Q2
Solving 14/25 | SCI_Q3
Solving 15/25 | SCI_Q4
Solving 16/25 | SCI_Q5
Solving 17/25 | SCI_Q6
Solving 18/25 | SCI_Q7
Solving 19/25 | MATH_Q1
Solving 20/25 | MATH_Q2
Solving 21/25 | MATH_Q3
Solving 22/25 | MATH_Q4
Solving 23/25 | MATH_Q5
Solving 24/25 | MATH_Q6
Solving 25/25 | MATH_Q7


,retriever,question_id,competition,question_type,selected_option,raw_response
0,DuckDuckGo,AHP_Q1,Ancient History and Politics,direct_entity,2.0,2.
1,DuckDuckGo,AHP_Q2,Ancient History and Politics,artifact_material,1.0,1.
2,DuckDuckGo,AHP_Q3,Ancient History and Politics,named_historical_figure,3.0,3. Demet
3,DuckDuckGo,AHP_Q4,Ancient History and Politics,conceptual_definition,1.0,1.
4,DuckDuckGo,AHP_Q5,Ancient History and Politics,comparison_reasoning,3.0,3.
5,DuckDuckGo,AHP_Q6,Ancient History and Politics,relationship_reasoning,2.0,2.
6,DuckDuckGo,ENT_Q1,Entertainment,music_terminology,2.0,2.
7,DuckDuckGo,ENT_Q2,Entertainment,music_voice_classification,1.0,1.
8,DuckDuckGo,ENT_Q3,Entertainment,celebrity_history,0.0,0.
9,DuckDuckGo,ENT_Q4,Entertainment,media_history,1.0,1.


In [ ]:
ground_truth_answers = {
    "AHP_Q1": 2,
    "AHP_Q2": 1,
    "AHP_Q3": 3,
    "AHP_Q4": 1,
    "AHP_Q5": 3,
    "AHP_Q6": 2,

    "ENT_Q1": 2,
    "ENT_Q2": 1,
    "ENT_Q3": 0,
    "ENT_Q4": 1,
    "ENT_Q5": 2,

    "SCI_Q1": 2,
    "SCI_Q2": 1,
    "SCI_Q3": 1,
    "SCI_Q4": 3,
    "SCI_Q5": 3,
    "SCI_Q6": 0,
    "SCI_Q7": 2,

    "MATH_Q1": 3,
    "MATH_Q2": 0,
    "MATH_Q3": 1,
    "MATH_Q4": 2,
    "MATH_Q5": 1,
    "MATH_Q6": 0,
    "MATH_Q7": 2
}

In [ ]:
# map ground-truth answers
ddg_rag_solver_df["correct_option"] = (
    ddg_rag_solver_df["question_id"]
    .map(ground_truth_answers)
)

# correctness
ddg_rag_solver_df["is_correct"] = (
    ddg_rag_solver_df["selected_option"]
    == ddg_rag_solver_df["correct_option"]
)

# per-competition accuracy
ddg_rag_competition_results_df = (
    ddg_rag_solver_df
    .groupby("competition")
    .agg(
        total_questions=("question_id", "count"),
        correct_answers=("is_correct", "sum")
    )
    .reset_index()
)

ddg_rag_competition_results_df["accuracy"] = (
    ddg_rag_competition_results_df["correct_answers"]
    / ddg_rag_competition_results_df["total_questions"]
)

# overall accuracy
overall_accuracy = ddg_rag_solver_df["is_correct"].mean()

print(f"Overall Accuracy: {overall_accuracy:.2%}")

ddg_rag_competition_results_df

Overall Accuracy: 68.00%


,competition,total_questions,correct_answers,accuracy
0,Ancient History and Politics,6,6,1.000000
1,Entertainment,5,5,1.000000
2,Maths,7,0,0.000000
3,Science and Nature,7,6,0.857143


### LLM Only

In [ ]:
# LLM-only baseline: solve questions without retrieved evidence

import re
import pandas as pd
import time

def build_llm_only_solver_prompt(question_item):
    options_text = "\n".join([
        f"{i}. {option}"
        for i, option in enumerate(question_item["options"])
    ])

    domain_context = DOMAIN_SOLVER_CONTEXT.get(
        question_item["competition"],
        "The question belongs to a general knowledge quiz category."
    )

    prompt = f"""
You are answering a multiple-choice quiz question.

Competition/domain:
{question_item["competition"]}

Domain guidance:
{domain_context}

Question:
{question_item["question"]}

Options:
{options_text}

Rules:
- Choose the single best option.
- Answer using ONLY the option number.
- The option number must be one of: 0, 1, 2, or 3.
- Do NOT explain.
- Do NOT write any words.
- Output a single digit only.

Answer:
"""
    return prompt


llm_only_solver_rows = []

for idx, question_item in enumerate(retrieval_test_questions):

    question_id = question_item["id"]

    print(f"Solving {idx+1}/{len(retrieval_test_questions)} | {question_id}")

    prompt = build_llm_only_solver_prompt(question_item)

    raw_response = generate_mistral_response(
        prompt,
        max_new_tokens=5,
        temperature=0.0
    )

    match = re.search(r"[0-3]", raw_response)
    selected_option = int(match.group()) if match else None

    llm_only_solver_rows.append({
        "retriever": "No Retrieval",
        "question_id": question_id,
        "competition": question_item["competition"],
        "question_type": question_item["question_type"],
        "selected_option": selected_option,
        "raw_response": raw_response
    })

    time.sleep(0.2)

llm_only_solver_df = pd.DataFrame(llm_only_solver_rows)

llm_only_solver_df

Solving 1/25 | AHP_Q1
Solving 2/25 | AHP_Q2
Solving 3/25 | AHP_Q3
Solving 4/25 | AHP_Q4
Solving 5/25 | AHP_Q5
Solving 6/25 | AHP_Q6
Solving 7/25 | ENT_Q1
Solving 8/25 | ENT_Q2
Solving 9/25 | ENT_Q3
Solving 10/25 | ENT_Q4
Solving 11/25 | ENT_Q5
Solving 12/25 | SCI_Q1
Solving 13/25 | SCI_Q2
Solving 14/25 | SCI_Q3
Solving 15/25 | SCI_Q4
Solving 16/25 | SCI_Q5
Solving 17/25 | SCI_Q6
Solving 18/25 | SCI_Q7
Solving 19/25 | MATH_Q1
Solving 20/25 | MATH_Q2
Solving 21/25 | MATH_Q3
Solving 22/25 | MATH_Q4
Solving 23/25 | MATH_Q5
Solving 24/25 | MATH_Q6
Solving 25/25 | MATH_Q7


,retriever,question_id,competition,question_type,selected_option,raw_response
0,No Retrieval,AHP_Q1,Ancient History and Politics,direct_entity,2,2.
1,No Retrieval,AHP_Q2,Ancient History and Politics,artifact_material,3,3. Bronze
2,No Retrieval,AHP_Q3,Ancient History and Politics,named_historical_figure,3,3. Demet
3,No Retrieval,AHP_Q4,Ancient History and Politics,conceptual_definition,1,1.
4,No Retrieval,AHP_Q5,Ancient History and Politics,comparison_reasoning,3,3.
5,No Retrieval,AHP_Q6,Ancient History and Politics,relationship_reasoning,2,2.
6,No Retrieval,ENT_Q1,Entertainment,music_terminology,2,2.
7,No Retrieval,ENT_Q2,Entertainment,music_voice_classification,1,1.
8,No Retrieval,ENT_Q3,Entertainment,celebrity_history,0,0.
9,No Retrieval,ENT_Q4,Entertainment,media_history,1,1.


In [ ]:
# compare LLM-only answers with ground truth

llm_only_solver_df["correct_option"] = (
    llm_only_solver_df["question_id"]
    .map(ground_truth_answers)
)

llm_only_solver_df["is_correct"] = (
    llm_only_solver_df["selected_option"]
    == llm_only_solver_df["correct_option"]
)

# per-competition accuracy
llm_only_competition_results_df = (
    llm_only_solver_df
    .groupby("competition")
    .agg(
        total_questions=("question_id", "count"),
        correct_answers=("is_correct", "sum")
    )
    .reset_index()
)

llm_only_competition_results_df["accuracy"] = (
    llm_only_competition_results_df["correct_answers"]
    / llm_only_competition_results_df["total_questions"]
)

# overall accuracy
overall_llm_only_accuracy = (
    llm_only_solver_df["is_correct"].mean()
)

print(f"Overall LLM-Only Accuracy: {overall_llm_only_accuracy:.2%}")

llm_only_competition_results_df

Overall LLM-Only Accuracy: 68.00%


,competition,total_questions,correct_answers,accuracy
0,Ancient History and Politics,6,5,0.833333
1,Entertainment,5,5,1.000000
2,Maths,7,1,0.142857
3,Science and Nature,7,6,0.857143


### Comparing RAG with LLM


In [ ]:
comparison_df = (
    ddg_rag_solver_df[
        [
            "question_id",
            "competition",
            "selected_option",
            "is_correct"
        ]
    ]
    .rename(columns={
        "selected_option": "rag_selected_option",
        "is_correct": "rag_correct"
    })
    .merge(
        llm_only_solver_df[
            [
                "question_id",
                "selected_option",
                "is_correct"
            ]
        ].rename(columns={
            "selected_option": "llm_only_selected_option",
            "is_correct": "llm_only_correct"
        }),
        on="question_id",
        how="inner"
    )
)

comparison_df["retrieval_helped"] = (
    (comparison_df["rag_correct"] == True)
    &
    (comparison_df["llm_only_correct"] == False)
)

comparison_df["retrieval_hurt"] = (
    (comparison_df["rag_correct"] == False)
    &
    (comparison_df["llm_only_correct"] == True)
)

comparison_df

,question_id,competition,rag_selected_option,rag_correct,llm_only_selected_option,llm_only_correct,retrieval_helped,retrieval_hurt
0,AHP_Q1,Ancient History and Politics,2.0,True,2,True,False,False
1,AHP_Q2,Ancient History and Politics,1.0,True,3,False,True,False
2,AHP_Q3,Ancient History and Politics,3.0,True,3,True,False,False
3,AHP_Q4,Ancient History and Politics,1.0,True,1,True,False,False
4,AHP_Q5,Ancient History and Politics,3.0,True,3,True,False,False
5,AHP_Q6,Ancient History and Politics,2.0,True,2,True,False,False
6,ENT_Q1,Entertainment,2.0,True,2,True,False,False
7,ENT_Q2,Entertainment,1.0,True,1,True,False,False
8,ENT_Q3,Entertainment,0.0,True,0,True,False,False
9,ENT_Q4,Entertainment,1.0,True,1,True,False,False


In [ ]:
comparison_summary = {
    "both_correct": (
        (comparison_df["rag_correct"] == True)
        &
        (comparison_df["llm_only_correct"] == True)
    ).sum(),

    "rag_only_correct": (
        (comparison_df["rag_correct"] == True)
        &
        (comparison_df["llm_only_correct"] == False)
    ).sum(),

    "llm_only_correct": (
        (comparison_df["rag_correct"] == False)
        &
        (comparison_df["llm_only_correct"] == True)
    ).sum(),

    "both_wrong": (
        (comparison_df["rag_correct"] == False)
        &
        (comparison_df["llm_only_correct"] == False)
    ).sum()
}

comparison_summary

{'both_correct': np.int64(16),
 'rag_only_correct': np.int64(1),
 'llm_only_correct': np.int64(1),
 'both_wrong': np.int64(7)}

## LLM + DuckDuckGo + Competetion

In [ ]:
def get_question_field(question_obj, field_name, default=None):
    if isinstance(question_obj, dict):
        return question_obj.get(field_name, default)
    return getattr(question_obj, field_name, default)


def duckduckgo_mistral_rag_solver(question_payload):
    """
    Real-game RAG solver using:
    DuckDuckGo retrieval + local Mistral answering
    Works with both dict payloads and Question objects.
    """

    question_text = get_question_field(question_payload, "question")
    if question_text is None:
        question_text = get_question_field(question_payload, "text")

    options = get_question_field(question_payload, "options", [])

    question_item = {
        "id": get_question_field(question_payload, "id", "LIVE_QUESTION"),
        "competition": get_question_field(question_payload, "competition", "General"),
        "question": question_text,
        "options": options
    }

    retrieved_results = retrieve_from_duckduckgo(
        question_item=question_item,
        max_results=3
    )

    retrieved_rows = [
        {
            "title": r.title,
            "url": r.url,
            "content": r.content
        }
        for r in retrieved_results
    ]

    prompt = build_rag_solver_number_only_prompt(
        question_item=question_item,
        retrieved_rows=retrieved_rows
    )

    raw_response = generate_mistral_response(
        prompt,
        max_new_tokens=5,
        temperature=0.0
    )

    match = re.search(r"[0-3]", raw_response)
    selected_option = int(match.group()) if match else 0

    print("Model response:", raw_response)
    print("Selected option:", selected_option)

    return selected_option

### Entertainment (15 questions)


In [ ]:
# Choose a competition ID
comp_id = 0

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=duckduckgo_mistral_rag_solver,
    solver_name="duckduckgo_mistral_rag"
)



--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What term describes a film that lampoons other film genres or films by imitating their styles?

Options:
0. Melodrama
1. Satire
2. Comedy
3. Parody

Answer:

Time remaining: 29.9s
Model response: 3.
Selected option: 3

Selected answer: 3
Decision time: 3.80s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What is the fundamental principle of a film portal that relates to its content organization?

Options:
0. It lists all movies in alphabetical order.
1. It features only the most popular films of the year.
2. It categorizes and provides links to films based on

###  Ancient History and Politics (15 questions)


In [ ]:
# Choose a competition ID
comp_id = 1

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=duckduckgo_mistral_rag_solver,
    solver_name="duckduckgo_mistral_rag"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
According to Hittite documents, which term is believed to be related to the name of the Achaeans, and is mentioned in the Tawagalawa letter?

Options:
0. Ahhiyawa
1. Madduwatta
2. Tawagalawa
3. Wilusa

Answer:

Time remaining: 29.9s
Model response: 0.
Selected option: 0

Selected answer: 0
Decision time: 3.94s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What is the primary purpose of the Colosseum in ancient Rome?

Options:
0. To be used as a fortress for the Roman military
1. To act as a prison for political dissidents
2. To serve as a marketplace for tra

###  Science and Nature (15 questions)

In [ ]:
# Choose a competition ID
comp_id = 2

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=duckduckgo_mistral_rag_solver,
    solver_name="duckduckgo_mistral_rag"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
Warm water contains less dissolved oxygen than colder water. Which population will be most affected by a long period of hot, dry days?

Options:
0. plants in streams
1. fish in ponds
2. plants in lakes
3. fish in oceans

Answer:

Time remaining: 29.9s
Model response: 1.
Selected option: 1

Selected answer: 1
Decision time: 3.09s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
Which tool is best used to collect information about the weather?

Options:
0. stopwatch
1. collecting net
2. ruler
3. thermometer

Answer:

Time remaining: 29.9s
Model response: 3.
Selec

### Trying other prompt

In [ ]:
def build_rag_solver_number_only_prompt(question_item, retrieved_rows):
    options_text = "\n".join([
        f"{i}. {option}"
        for i, option in enumerate(question_item["options"])
    ])

    domain_context = DOMAIN_SOLVER_CONTEXT.get(
        question_item.get("competition", "General"),
        "The question belongs to a general knowledge quiz category."
    )

    evidence_blocks = []

    for idx, row in enumerate(retrieved_rows, start=1):
        evidence_blocks.append(
            f"""
Evidence {idx}
Title: {row["title"]}
Content:
{row["content"]}
"""
        )

    evidence_text = "\n".join(evidence_blocks)

    prompt = f"""
You are answering a multiple-choice quiz question.

Competition/domain:
{question_item.get("competition", "General")}

Domain guidance:
{domain_context}

Question:
{question_item["question"]}

Options:
{options_text}

Retrieved evidence:
{evidence_text}

Decision rules:
- Compare all 4 options carefully.
- Eliminate options contradicted by the question or evidence.
- If retrieved evidence is irrelevant or noisy, rely on reasoning from the question and options.
- For science/math/logic questions, reason from principles instead of blindly trusting retrieved text.
- Choose the single best option.
- Answer using ONLY the option number.
- The option number must be one of: 0, 1, 2, or 3.
- Do NOT explain.
- Do NOT write any words.
- Output a single digit only.

Answer:
"""
    return prompt

### Entertainment (15 questions)


In [ ]:
# Choose a competition ID
comp_id = 0

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=duckduckgo_mistral_rag_solver,
    solver_name="duckduckgo_mistral_rag"
)



--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What was the primary reason for Marilyn Monroe's sudden death in 1962?

Options:
0. She was involved in a plane crash.
1. She was murdered by a conspiracy.
2. She died in a car accident.
3. She accidentally overdosed on barbiturates.

Answer:

Time remaining: 29.9s
Model response: 3.
Selected option: 3

Selected answer: 3
Decision time: 4.26s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
Which of the following best describes Coppola's approach to directing 'The Godfather'?

Options:
0. He followed the novel closely without changes
1. He aimed for a broad, hi

###  Ancient History and Politics (15 questions)


In [ ]:
# Choose a competition ID
comp_id = 1

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=duckduckgo_mistral_rag_solver,
    solver_name="duckduckgo_mistral_rag"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
According to Hittite documents, which term is believed to be related to the name of the Achaeans, and is mentioned in the Tawagalawa letter?

Options:
0. Madduwatta
1. Tawagalawa
2. Ahhiyawa
3. Wilusa

Answer:

Time remaining: 29.9s
Model response: 2.
Selected option: 2

Selected answer: 2
Decision time: 6.10s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
What was the fundamental principle of ancient Egyptian civilization that guided their daily lives and society?

Options:
0. The construction of pyramids
1. The practice of mummification
2. The worship of mu

###  Science and Nature (15 questions)

In [ ]:
# Choose a competition ID
comp_id = 2

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=duckduckgo_mistral_rag_solver,
    solver_name="duckduckgo_mistral_rag"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
Abnormal cell division may result in

Options:
0. tissue repair
1. cancer
2. metamorphosis
3. disease prevention

Answer:

Time remaining: 29.9s
Model response: 1.
Selected option: 1

Selected answer: 1
Decision time: 3.77s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
Which protein is the defining component of Lewy bodies?

Options:
0. TDP-43
1. Tau
2. Alpha-synuclein (α-synuclein)
3. Beta-amyloid

Answer:

Time remaining: 29.9s
Model response: 2.
Selected option: 2

Selected answer: 2
Decision time: 3.25s
 CORRECT!
 Earned so far: $200.00

--- Level 3 ---


### Real Competition Evaluation — DuckDuckGo + Local Mistral RAG

In this section, the retrieval-augmented pipeline was evaluated on live competition gameplay using:

- DuckDuckGo web retrieval
- Local Mistral-7B-Instruct model
- Retrieval-Augmented Generation (RAG)
- Multiple-choice constrained answering

### Key Characteristics

The solver:
1. Retrieves external evidence from the web,
2. Injects retrieved context into the prompt,
3. Uses the local LLM to reason over:
   - question,
   - options,
   - retrieved evidence,
4. Outputs only the selected option number.

### Main Observations

The system performed strongly on:
- factual recall,
- entertainment/media knowledge,
- history and politics,
- biology and general science.

The pipeline successfully reached advanced competition levels and demonstrated:
- stable retrieval behavior,
- low latency,
- strong factual grounding,
- robust answer formatting.

### Important Findings

Several failure cases revealed important RAG limitations:

- Retrieval noise can occasionally distract the model,
- Some questions require deeper reasoning rather than factual lookup,
- Scientific and logical questions may still fail despite retrieval support,
- Incorrect or weak evidence can negatively influence final decisions.

These experiments highlight an important distinction between:

- retrieval capability,
- reasoning capability,
- and tool-augmented problem solving.

### Conclusion

The DuckDuckGo + Mistral pipeline demonstrated that lightweight web retrieval combined with a local open-source LLM can achieve strong real-time quiz performance with relatively low infrastructure complexity.

The next stage focuses on augmenting the system with dedicated mathematical and computational tools to improve reasoning-intensive question handling.

# groq

In [ ]:
# !pip install -q -U langchain-groq
!pip install -q -U groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 1.1.2 requires groq<1.0.0,>=0.30.0, but you have groq 1.2.0 which is incompatible.


In [ ]:
import os
from google.colab import userdata
from groq import Groq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

groq_client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Groq client ready")

Groq client ready


In [ ]:
def groq_invoke(prompt, model="llama-3.3-70b-versatile"):
    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are a precise multiple-choice competition assistant. Return only the option number."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0,
        max_tokens=20,
    )
    return response.choices[0].message.content

In [ ]:
raw_answer = groq_invoke("Solve: 2x + 5 = 17. Return only the value of x.")
print(raw_answer)

6


### Test on the bank of questions

In [ ]:
import re
import pandas as pd
from tqdm import tqdm

def format_mcq_prompt(q):
    options_text = "\n".join(
        [f"{i}. {opt}" for i, opt in enumerate(q["options"])]
    )

    return f"""
You are answering a multiple-choice competition question.

Rules:
- Choose the best answer.
- Return ONLY the option number: 0, 1, 2, or 3.
- Do not explain.

Question:
{q["question"]}

Options:
{options_text}

Answer:
""".strip()


def extract_option(text):
    text = str(text).strip()

    match = re.search(r"\b[0-3]\b", text)
    if match:
        return int(match.group())

    return None


In [ ]:


results = []

for q in tqdm(retrieval_test_questions):
    prompt = format_mcq_prompt(q)

    raw_answer = groq_invoke(
        prompt,
        model="llama-3.3-70b-versatile"
    )

    predicted_option = extract_option(raw_answer)

    results.append({
        "id": q["id"],
        "competition": q["competition"],
        "question_type": q["question_type"],
        "retrieval_needed": q["retrieval_needed"],
        "tool_needed": q["tool_needed"],
        "expected_option": q["expected_option"],
        "predicted_option": predicted_option,
        "is_correct": predicted_option == q["expected_option"],
        "raw_answer": raw_answer
    })

df_results = pd.DataFrame(results)

accuracy = df_results["is_correct"].mean()

print(f"Baseline accuracy: {accuracy:.2%}")
display(df_results)

100%|██████████| 25/25 [00:02<00:00,  8.63it/s]


Baseline accuracy: 76.00%


,id,competition,question_type,retrieval_needed,tool_needed,expected_option,predicted_option,is_correct,raw_answer
0,AHP_Q1,Ancient History and Politics,direct_entity,True,None,2,2,True,2
1,AHP_Q2,Ancient History and Politics,artifact_material,True,None,1,3,False,3
2,AHP_Q3,Ancient History and Politics,named_historical_figure,True,None,3,3,True,3
3,AHP_Q4,Ancient History and Politics,conceptual_definition,True,None,1,1,True,1
4,AHP_Q5,Ancient History and Politics,comparison_reasoning,True,None,3,3,True,3
5,AHP_Q6,Ancient History and Politics,relationship_reasoning,True,None,2,2,True,2
6,ENT_Q1,Entertainment,music_terminology,True,None,2,2,True,2
7,ENT_Q2,Entertainment,music_voice_classification,True,None,1,1,True,1
8,ENT_Q3,Entertainment,celebrity_history,True,None,0,0,True,0
9,ENT_Q4,Entertainment,media_history,True,None,1,1,True,1


### Interate groq & Langchain

In [ ]:
!pip install -q -U langchain-openai langgraph langchain-tavily

In [ ]:
!pip install -q \
langchain==0.3.25 \
langchain-core==0.3.59 \
langchain-openai==0.3.16 \
langsmith==0.3.42 \
openai

In [ ]:
import os
from google.colab import userdata
from langchain_openai import ChatOpenAI

groq_key = userdata.get("GROQ_API_KEY")

llm = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=groq_key,
    model="llama-3.3-70b-versatile",
    temperature=0,
)

print("Groq via LangChain is ready")

Groq via LangChain is ready


In [ ]:
test_q = retrieval_test_questions[0]
prompt = format_mcq_prompt(test_q)

response = llm.invoke(prompt)

print(response.content)
print("Expected:", test_q["expected_option"])

2
Expected: 2


In [ ]:
import re
import pandas as pd
from tqdm import tqdm

def format_mcq_prompt(q):
    options_text = "\n".join(
        [f"{i}. {opt}" for i, opt in enumerate(q["options"])]
    )

    return f"""
You are answering a multiple-choice competition question.

Rules:
- Choose the best answer.
- Return ONLY the option number: 0, 1, 2, or 3.
- Do not explain.

Question:
{q["question"]}

Options:
{options_text}

Answer:
""".strip()


def extract_option(text):
    text = str(text).strip()

    match = re.search(r"\b[0-3]\b", text)
    if match:
        return int(match.group())

    return None


results = []

for q in tqdm(retrieval_test_questions):

    prompt = format_mcq_prompt(q)

    response = llm.invoke(prompt)
    raw_answer = response.content

    predicted_option = extract_option(raw_answer)

    results.append({
        "id": q["id"],
        "competition": q["competition"],
        "question_type": q["question_type"],
        "retrieval_needed": q["retrieval_needed"],
        "tool_needed": q["tool_needed"],
        "expected_option": q["expected_option"],
        "predicted_option": predicted_option,
        "is_correct": predicted_option == q["expected_option"],
        "raw_answer": raw_answer
    })

df_results = pd.DataFrame(results)

accuracy = df_results["is_correct"].mean()

print(f"Baseline accuracy: {accuracy:.2%}")

display(df_results)

100%|██████████| 25/25 [00:03<00:00,  8.16it/s]


Baseline accuracy: 76.00%


,id,competition,question_type,retrieval_needed,tool_needed,expected_option,predicted_option,is_correct,raw_answer
0,AHP_Q1,Ancient History and Politics,direct_entity,True,None,2,2,True,2
1,AHP_Q2,Ancient History and Politics,artifact_material,True,None,1,3,False,3
2,AHP_Q3,Ancient History and Politics,named_historical_figure,True,None,3,3,True,3
3,AHP_Q4,Ancient History and Politics,conceptual_definition,True,None,1,1,True,1
4,AHP_Q5,Ancient History and Politics,comparison_reasoning,True,None,3,3,True,3
5,AHP_Q6,Ancient History and Politics,relationship_reasoning,True,None,2,2,True,2
6,ENT_Q1,Entertainment,music_terminology,True,None,2,2,True,2
7,ENT_Q2,Entertainment,music_voice_classification,True,None,1,1,True,1
8,ENT_Q3,Entertainment,celebrity_history,True,None,0,0,True,0
9,ENT_Q4,Entertainment,media_history,True,None,1,1,True,1


In [ ]:
!pip install -q -U langgraph

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-groq 1.1.2 requires groq<1.0.0,>=0.30.0, but you have groq 1.2.0 which is incompatible.
langchain 0.3.25 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.3.3 which is incompatible.
langchain 0.3.25 requires langsmith<0.4,>=0.1.17, but you have langsmith 0.8.3 which is incompatible.
langchain-openai 0.3.16 requires langchain-core<1.0.0,>=0.3.58, but you have langchain-core 1.3.3 which is incompatible.
langchain-huggingface 0.1.2 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 1.3.3 which is incompatible.
langchain-text-splitters 0.3.8 requires langchain-core<1.0.0,>=0.3.51, but you have langchain-core 1.3.3 which is incompatible.
langchain-tavily 0.2.18 requires langchain<2.0.0,>=1.0.0, but you have langchain 0.3.25 which is incompatible.
langchain-classic 1.0.7 re

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def python_math_tool(code: str) -> str:
    """
    Execute Python code for math reasoning.
    Use this for arithmetic, statistics, equations, checking answer choices, and logical verification.
    Store the final result in a variable named answer.
    """
    try:
        safe_globals = {
            "__builtins__": {},
            "sum": sum,
            "min": min,
            "max": max,
            "len": len,
            "round": round,
            "abs": abs,
            "pow": pow,
            "range": range,
        }

        local_vars = {}
        exec(code, safe_globals, local_vars)

        if "answer" in local_vars:
            return str(local_vars["answer"])

        return str(local_vars)

    except Exception as e:
        return f"Error: {e}"


agent = create_react_agent(
    model=llm,
    tools=[python_math_tool]
)

print("Math agent ready")

In [ ]:
q = retrieval_test_questions[-1]
prompt = format_mcq_prompt(q)

response = agent.invoke({
    "messages": [
        ("system", "You are a math competition agent. Use the Python tool when calculation helps. Final answer must be only 0, 1, 2, or 3."),
        ("user", prompt)
    ]
})

print(response["messages"][-1].content)

In [ ]:
# Choose a competition ID
comp_id = 3

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=duckduckgo_mistral_rag_solver,
    solver_name="math_mistral_solver"
)

### Trying groq with other competetions

In [ ]:
import re

def groq_langchain_solver(question):

    formatted_q = format_question(question)

    prompt = f"""
You are answering a multiple-choice competition question.

Rules:
- Return ONLY one number: 0, 1, 2, or 3.
- Do not explain.
- Do not add text.

{formatted_q}

Answer:
""".strip()

    response = llm.invoke(prompt)
    raw_answer = response.content.strip()

    match = re.search(r"\b[0-3]\b", raw_answer)

    if match:
        return int(match.group())

    print("Could not parse model answer:", raw_answer)
    return 0

In [ ]:
import time
# Choose a competition ID
comp_id = 0

game = client.game.start(competition_id=comp_id)

play_game_auto(
    game,
    solver_function=groq_langchain_solver,
    solver_name="groq_langchain"
)


--- Level 1 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- Do NOT explain
- Do NOT write text
- Output a single number only

Question:
How does Al Pacino's role as Michael Corleone in The Godfather differ from his role as Tony Montana in Scarface in terms of character development?

Options:
0. Michael Corleone is a street punk, while Tony Montana is a high-level mafia boss.
1. Michael Corleone is a rising criminal, while Tony Montana is a fall from grace.
2. Michael Corleone is a quiet and calculating leader, while Tony Montana is more impulsive and violent.
3. Michael Corleone is a young up-and-comer, while Tony Montana is an older, wiser leader.

Answer:

Time remaining: 29.9s

Selected answer: 2
Decision time: 0.07s
 CORRECT!
 Earned so far: $100.00

--- Level 2 ---
You are answering a multiple choice question.

Choose the correct option.

Rules:
- Answer using ONLY the number (0, 1, 2, or 3)
- 